In [ ]:
import os
import json
import numpy as np
from datetime import datetime, timedelta
from scipy.stats import truncnorm
import matplotlib.pyplot as plt
import random

def parse_datetime(dt_str):
    """
    Parse a GMT datetime string and convert to local (America/Los_Angeles, GMT-8).
    """
    dt_gmt = datetime.strptime(dt_str, "%a, %d %b %Y %H:%M:%S %Z")
    return dt_gmt - timedelta(hours=8)

def compute_available_minutes(st, et):
    """
    Compute minutes between local start and end datetimes,
    wrapping past midnight if necessary.
    """
    if et >= st:
        delta = et - st
    else:
        delta = (et + timedelta(days=1)) - st
    return delta.total_seconds() / 60

def launch_CMAs(charging_profiles, control_params, compromised_indices):
    """
    Apply CMA Types 1–6 to the input charging profiles.
    
    charging_profiles: dict of numpy arrays 'st', 'et', 'd'
    control_params: dict with 'Catt', 'Chav', 'Pl', 'dt'
    compromised_indices: dict with lists of indices:
        'Nhd' (Type-1), 'Nhe' (Type-2), 'Nhs' (Type-3),
        'N4'  (Type-4), 'N5'  (Type-5), 'N6'  (Type-6)
    """
    dt = control_params['dt']
    ACRatt = control_params['Catt'] * control_params['Chav']
    
    # Copy originals
    new_profiles = {
        'st': charging_profiles['st'].copy(),
        'et': charging_profiles['et'].copy(),
        'd':  charging_profiles['d'].copy(),
        'disc': charging_profiles['disc'].copy()
    }
    n = len(charging_profiles['st'])
    
    # Precompute available slots and base rates
    available_steps = np.zeros(n)
    for i in range(n):
        mins = compute_available_minutes(charging_profiles['st'][i],
                                         charging_profiles['et'][i])
        available_steps[i] = mins / dt
    
    Ch_all = np.zeros(n)
    for i in range(n):
        steps = available_steps[i]
        d_val = charging_profiles['d'][i]
        Ch_all[i] = d_val * (60/dt) / steps if steps > 0 else 0

    mu1, sigma1 = 0.9, 0.1
    mu2, sigma2 = 0.9, 0.1
    mu3, sigma3 = 0.9, 0.1
    mu4, sigma4 = 0.8, 0.1


    # truncation bounds in standard‐normal units:
    #   a = (lower_bound - mu) / sigma
    #   b = (upper_bound - mu) / sigma
    a1, b1 = (0 - mu1) / sigma1, (1 - mu1) / sigma1
    a2, b2 = (0 - mu2) / sigma2, (1 - mu2) / sigma2
    a3, b3 = (0 - mu3) / sigma3, (1 - mu3) / sigma3
    a4, b4 = (0 - mu4) / sigma4, (1 - mu4) / sigma4

    # --- Type‑1: demand manipulation only ---
    for i in compromised_indices.get('Nhd', []):
        steps = available_steps[i]
        Ch = Ch_all[i]
        d_val = charging_profiles['d'][i]
        # new_d = ACRatt * total_minutes/60 - Ch*(dt/60) + original_d
        #new_d = ACRatt * (steps*dt/60) - Ch*(dt/60) + d_val
        x = truncnorm.rvs(a1, b1, loc=mu1, scale=sigma1)
        new_d = x*ACRatt * ((steps-1)*dt/60) + d_val
        new_profiles['d'][i] = new_d
        # st & et unchanged :contentReference[oaicite:0]{index=0}&#8203;:contentReference[oaicite:1]{index=1}


    # --- Type‑2: demand + end‑time shift (earlier) ---
    for i in compromised_indices.get('Nhe', []):
        steps = available_steps[i]
        Ch = Ch_all[i]
        d_val = charging_profiles['d'][i]
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        # compute new demand
        #new_d = ACRatt * (steps*dt/60) - Ch*(dt/60) + d_val
        #new_profiles['d'][i] = new_d
        # shift end time earlier
        x = truncnorm.rvs(a2, b2, loc=mu2, scale=sigma2)
        new_steps = ((d_val - Ch*(dt/60))/ (x*ACRatt+Ch))*(60/dt) +1
        delta = abs(steps - new_steps) * dt
        new_profiles['et'][i] = et - timedelta(minutes=delta)

    # --- Type‑3: demand + start‑time shift (later) ---
    for i in compromised_indices.get('Nhs', []):
        steps = available_steps[i]
        Ch = Ch_all[i]
        d_val = charging_profiles['d'][i]
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        #new_d = ACRatt * (steps*dt/60) - Ch*(dt/60) + d_val
        #new_profiles['d'][i] = new_d
        #new_steps = (d_val / new_d) * steps if new_d != 0 else steps
        #new_steps = ((d_val - Ch*(dt/60))/ (ACRatt+Ch))*(60/dt)
        x = truncnorm.rvs(a3, b3, loc=mu3, scale=sigma3)
        new_steps = ((d_val - Ch*(dt/60))/ (x*ACRatt+Ch))*(60/dt)
        delta = abs(steps - new_steps) * dt
        new_profiles['st'][i] = st + timedelta(minutes=delta)

    # --- Type‑4: mixed demand + end‑time (|Nhd|,|Nhe| ≥ 0.4|Nh|) ---
    # desired parameters

    for i in compromised_indices.get('N4', []):
        steps = available_steps[i]
        Ch = Ch_all[i]
        d_val = charging_profiles['d'][i]
        et = charging_profiles['et'][i]
        # draw a single sample
        x = truncnorm.rvs(a4, b4, loc=mu4, scale=sigma4)
        ACRxatt = x * control_params['Catt'] * control_params['Chav']
        #new_d = ACRatt * (steps*dt/60) - Ch*(dt/60) + d_val
        new_d = ACRxatt * ((steps-1)*dt/60) + d_val
        new_profiles['d'][i] = new_d
        #new_steps = (d_val / new_d) * steps if new_d != 0 else steps
        new_steps = (new_d - Ch*(dt/60)) / (ACRatt+Ch) * (60/dt) +1
        delta = abs(steps - new_steps) * dt
        new_profiles['et'][i] = et - timedelta(minutes=delta)
    # --- Type‑5: mixed demand + start‑time (|Nhd|,|Nhs| ≥ 0.4|Nh|) ---
    for i in compromised_indices.get('N5', []):
        steps = available_steps[i]
        Ch = Ch_all[i]
        d_val = charging_profiles['d'][i]
        st = charging_profiles['st'][i]
        x = truncnorm.rvs(a4, b4, loc=mu4, scale=sigma4)
        ACRxatt = x * control_params['Catt'] * control_params['Chav']
        #new_d = ACRatt * (steps*dt/60) - Ch*(dt/60) + d_val
        new_d = ACRxatt * ((steps-1)*dt/60) + d_val
        new_profiles['d'][i] = new_d
        #new_steps = (d_val / new_d) * steps if new_d != 0 else steps
        new_steps = (new_d - Ch*(dt/60)) / (ACRatt+Ch) * (60/dt)
        delta = abs(steps - new_steps) * dt
        new_profiles['st'][i] = st + timedelta(minutes=delta)

    # --- Type‑6: mix of all three (|Nhd|,|Nhe|,|Nhs| ≤ 0.4|Nh|) ---
    for i in compromised_indices.get('N6', []):
        steps = available_steps[i]
        Ch = Ch_all[i]
        d_val = charging_profiles['d'][i]
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        x = truncnorm.rvs(a4, b4, loc=mu4, scale=sigma4)
        ACRxatt = x * control_params['Catt'] * control_params['Chav']
        # demand
        new_d = ACRxatt * ((steps-1)*dt/60) + d_val
        new_profiles['d'][i] = new_d
        # end-time earlier
        new_steps = (new_d - Ch*(dt/60)) / (ACRatt+Ch) * (60/dt) + 1
        delta = abs(steps - new_steps) * dt
        st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
        et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
        if st < st_peek and et > et_peek:
            theta = (et - et_peek).total_seconds()/((et - st)-(et_peek - st_peek)).total_seconds()
            new_profiles['et'][i] = et - timedelta(minutes=(theta*delta))
            new_profiles['st'][i] = st + timedelta(minutes=((1-theta)*delta + dt))
        elif (st >= st_peek and et > et_peek) or (st >= (st_peek + (et_peek - st_peek)/2) and et <= et_peek):
            theta = 1
            new_profiles['et'][i] = et - timedelta(minutes=(theta*delta))
            new_profiles['st'][i] = st + timedelta(minutes=dt)
        elif (st < st_peek and et <= et_peek) or (st >= st_peek  and et <= (st_peek + (et_peek - st_peek)/2)):
            theta = 0
            new_profiles['st'][i] = st + timedelta(minutes=((1-theta)*delta + dt))
        elif st >= st_peek and st <= (st_peek + (et_peek - st_peek)/2) and et <= et_peek and et >=  (st_peek + (et_peek - st_peek)/2):
            theta = (et - (st_peek + (et_peek - st_peek)/2)).total_seconds()/(et - st).total_seconds()
            new_profiles['et'][i] = et - timedelta(minutes=(theta*delta))
        else:
            new_profiles['st'][i] = st + timedelta(minutes=dt)
    return new_profiles

def launch_CMAs_ta(charging_profiles, control_params, compromised_indices, ta:datetime):
    """
    Apply CMA Types 1–6 to the input charging profiles.
    
    charging_profiles: dict of numpy arrays 'st', 'et', 'd'
    control_params: dict with 'Catt', 'Chav', 'Pl', 'dt'
    compromised_indices: dict with lists of indices:
        'Nhd' (Type-1), 'Nhe' (Type-2), 'Nhs' (Type-3),
        'N4'  (Type-4), 'N5'  (Type-5), 'N6'  (Type-6)
    """
    dt = control_params['dt']
    ACRatt = control_params['Catt'] * control_params['Chav']
    
    
    # Copy originals
    new_profiles = {
        'st': charging_profiles['st'].copy(),
        'et': charging_profiles['et'].copy(),
        'd':  charging_profiles['d'].copy(),
        'disc': charging_profiles['disc'].copy()
    }
    n = len(charging_profiles['st'])
    
    # Precompute available slots and base rates
    available_steps = np.zeros(n)
    for i in range(n):
        mins = compute_available_minutes(charging_profiles['st'][i],
                                         charging_profiles['et'][i])
        available_steps[i] = mins / dt
    
    Ch_all = np.zeros(n)
    for i in range(n):
        steps = available_steps[i]
        d_val = charging_profiles['d'][i]
        Ch_all[i] = d_val * (60/dt) / steps if steps > 0 else 0

    mu1, sigma1 = 0.9, 0.1
    mu2, sigma2 = 0.9, 0.1
    mu3, sigma3 = 0.9, 0.1
    mu4, sigma4 = 0.8, 0.1


    # truncation bounds in standard‐normal units:
    #   a = (lower_bound - mu) / sigma
    #   b = (upper_bound - mu) / sigma
    a1, b1 = (0 - mu1) / sigma1, (1 - mu1) / sigma1
    a2, b2 = (0 - mu2) / sigma2, (1 - mu2) / sigma2
    a3, b3 = (0 - mu3) / sigma3, (1 - mu3) / sigma3
    a4, b4 = (0 - mu4) / sigma4, (1 - mu4) / sigma4

    # --- Type‑1: demand manipulation only ---
    for i in compromised_indices.get('N1_ta', []):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        ta1 = datetime.combine(st.date(), ta.time())
        #ta2 = datetime.combine(et.date(), ta.time())
        if st <= ta1 < et:
            ta = ta1
        else:
            continue            
        Ch = Ch_all[i]
        d_val = charging_profiles['d'][i]
        # new_d = ACRatt * total_minutes/60 - Ch*(dt/60) + original_d
        #new_d = ACRatt * (steps*dt/60) - Ch*(dt/60) + d_val
        x = truncnorm.rvs(a1, b1, loc=mu1, scale=sigma1)
        new_d = x*ACRatt * ((et - ta).total_seconds()/3600) + d_val
        new_profiles['d'][i] = new_d
        # st & et unchanged :contentReference[oaicite:0]{index=0}&#8203;:contentReference[oaicite:1]{index=1}


    # --- Type‑2: demand + end‑time shift (earlier) ---
    for i in compromised_indices.get('N2_ta', []):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        ta1 = datetime.combine(st.date(), ta.time())
        if st <= ta1 < et:
            ta = ta1
        else:
            continue    
        Ch = Ch_all[i]
        d_val = charging_profiles['d'][i]
        # compute new demand
        #new_d = ACRatt * (steps*dt/60) - Ch*(dt/60) + d_val
        #new_profiles['d'][i] = new_d
        # shift end time earlier
        x = truncnorm.rvs(a2, b2, loc=mu2, scale=sigma2)
        nT = d_val*((et - ta).total_seconds()/3600)/((et - st).total_seconds()/3600)/(x*ACRatt+Ch)
        new_profiles['et'][i] = ta + timedelta(hours=nT)

    # --- Type‑3: demand + start‑time shift (later) ---
    for i in compromised_indices.get('N3_ta', []):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        ta1 = datetime.combine(st.date(), ta.time())
        if st <= ta1 < et:
            ta = ta1
        else:
            continue     
        Ch = Ch_all[i]
        d_val = charging_profiles['d'][i]
        # compute new demand
        #new_d = ACRatt * (steps*dt/60) - Ch*(dt/60) + d_val
        #new_profiles['d'][i] = new_d
        # shift end time earlier
        x = truncnorm.rvs(a3, b3, loc=mu3, scale=sigma3)
        nT = d_val*((et - ta).total_seconds())/((et - st).total_seconds())/(x*ACRatt+Ch)
        new_profiles['st'][i] = et - timedelta(hours=nT)

    # --- Type‑4: mixed demand + end‑time (|Nhd|,|Nhe| ≥ 0.4|Nh|) ---
    # desired parameters

    for i in compromised_indices.get('N4_ta', []):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        ta1 = datetime.combine(st.date(), ta.time())
        if st <= ta1 < et:
            ta = ta1
        else:
            continue  
        Ch = Ch_all[i]
        d_val = charging_profiles['d'][i]
        # draw a single sample
        x = truncnorm.rvs(a4, b4, loc=mu4, scale=sigma4)
        ACRxatt = x * control_params['Catt'] * control_params['Chav']
        new_d = ACRxatt * ((et - ta).total_seconds()/3600) + d_val
        new_profiles['d'][i] = new_d
        nT = ((et - ta).total_seconds()/3600)*(Ch + ACRxatt)/(Ch + ACRatt)
        new_profiles['et'][i] = ta + timedelta(hours=nT)
    # --- Type‑5: mixed demand + start‑time (|Nhd|,|Nhs| ≥ 0.4|Nh|) ---
    for i in compromised_indices.get('N5_ta', []):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        ta1 = datetime.combine(st.date(), ta.time())
        if st <= ta1 < et:
            ta = ta1
        else:
            continue  
        Ch = Ch_all[i]
        d_val = charging_profiles['d'][i]
        # draw a single sample
        x = truncnorm.rvs(a4, b4, loc=mu4, scale=sigma4)
        ACRxatt = x * control_params['Catt'] * control_params['Chav']
        new_d = ACRxatt * ((et - ta).total_seconds()/3600) + d_val
        new_profiles['d'][i] = new_d
        nT = ((et - ta).total_seconds()/3600)*(Ch + ACRxatt)/(Ch + ACRatt)
        new_profiles['st'][i] = et - timedelta(hours=nT)

    # --- Type‑6: mix of all three (|Nhd|,|Nhe|,|Nhs| ≤ 0.4|Nh|) ---
    for i in compromised_indices.get('N6_ta', []):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        ta1 = datetime.combine(st.date(), ta.time())
        if st <= ta1 < et:
            ta = ta1
        else:
            continue  
        Ch = Ch_all[i]
        d_val = charging_profiles['d'][i]
        # draw a single sample
        x = truncnorm.rvs(a4, b4, loc=mu4, scale=sigma4)
        ACRxatt = x * control_params['Catt'] * control_params['Chav']
        new_d = ACRxatt * ((et - ta).total_seconds()/3600) + d_val
        new_profiles['d'][i] = new_d
        nT = ((et - ta).total_seconds()/3600)*(Ch + ACRxatt)/(Ch + ACRatt)
        delta = abs((et - ta).total_seconds()/3600 - nT)*60
        st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
        et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
        if ta < st_peek and et > et_peek:
            theta = (et - et_peek).total_seconds()/((et - ta)-(et_peek - st_peek)).total_seconds()
            new_profiles['et'][i] = et - timedelta(minutes=(theta*delta))
            new_profiles['st'][i] = ta + timedelta(minutes=((1-theta)*delta))
        elif (ta >= st_peek and et > et_peek) or (ta >= (st_peek + (et_peek - st_peek)/2) and et <= et_peek):
            theta = 1
            new_profiles['et'][i] = et - timedelta(minutes=(theta*delta))
            new_profiles['st'][i] = ta
        elif (ta < st_peek and et <= et_peek) or (ta >= st_peek  and et <= (st_peek + (et_peek - st_peek)/2)):
            theta = 0
            new_profiles['st'][i] = ta + timedelta(minutes=((1-theta)*delta))
        elif ta >= st_peek and ta <= (st_peek + (et_peek - st_peek)/2) and et <= et_peek and et >=  (st_peek + (et_peek - st_peek)/2):
            theta = (et - (st_peek + (et_peek - st_peek)/2)).total_seconds()/(et - ta).total_seconds()
            new_profiles['et'][i] = et - timedelta(minutes=(theta*delta))
        else:
            new_profiles['st'][i] = ta
    return new_profiles

import pandas as _pd

    # --- 稳健计算两个时间戳之间可用分钟数（处理跨日情况） ---
def minutes_between_safe(st_raw, et_raw):
        """
        返回 st_raw 到 et_raw 的分钟数（float），若 et < st 则视为跨到次日。
        支持 st_raw/et_raw 为 datetime / pd.Timestamp / np.datetime64 / 字符串等可解析为 pd.Timestamp 的类型。
        """
        s = _pd.Timestamp(st_raw)
        e = _pd.Timestamp(et_raw)
        if e < s:
            # 跨天，认为结束时间在次日
            e = e + _pd.Timedelta(days=1)
        td = e - s  # pd.Timedelta
        return td.total_seconds() / 60.0


def launch_CMAs_ta(charging_profiles, control_params, compromised_indices, ta):
    """
    Apply CMA Types 1–6 to the input charging profiles, supporting per-session ta.

    charging_profiles: dict with 'st','et','d','disc' (sequence-like)
    control_params: dict with 'Catt', 'Chav', 'Pl', 'dt'
    compromised_indices: dict mapping attack-key -> list of indices, keys expected:
        'N1_ta','N2_ta','N3_ta','N4_ta','N5_ta','N6_ta'
        (lists should be non-overlapping)
    ta: can be
        - a single datetime -> applied as original behavior (time-of-day combined with session date),
        - a sequence/array-like of datetimes length n (entry can be None/NaT for no attack),
        - a dict mapping session index -> datetime (only specified sessions have ta).
    Returns:
      new_profiles: same structure as input plus fields:
        'is_modified' (np.bool_ array),
        'ta_per_session' (np.datetime64[ns] array),
        'attack_type' (np.array of strings)
    """
    dt = control_params['dt']
    ACRatt = control_params['Catt'] * control_params['Chav']

    # copy originals (avoid modifying input)
    new_profiles = {
        'st': charging_profiles['st'].copy(),
        'et': charging_profiles['et'].copy(),
        'd':  charging_profiles['d'].copy(),
        'disc': charging_profiles['disc'].copy()
    }
    n = int(len(charging_profiles['st']))

    # Helper: convert ta param to per-session pandas Timestamp (or NaT)
    ta_per_session = np.full(n, np.datetime64('NaT'), dtype='datetime64[ns]')
    if isinstance(ta, dict):
        # dict: map provided entries
        for k, v in ta.items():
            idx = int(k)
            if v is None:
                ta_per_session[idx] = np.datetime64('NaT')
            else:
                ta_per_session[idx] = np.datetime64(pd.Timestamp(v))
    else:
        # if sequence-like (length n) then map element-wise; else single datetime -> use its time-of-day to combine with st.date()
        try:
            # try treating as sequence-like
            if hasattr(ta, "__len__") and (not isinstance(ta, datetime)):
                # sequence-like: may be shorter/longer; use as many as min(n,len(ta))
                L = len(ta)
                for i in range(min(n, L)):
                    v = ta[i]
                    ta_per_session[i] = np.datetime64(pd.NaT) if v is None else np.datetime64(pd.Timestamp(v))
                # others remain NaT
            else:
                # single datetime: we'll combine its time-of-day with each session's st.date() later when needed
                default_ta = pd.Timestamp(ta)
                # mark a special sentinel by leaving ta_per_session NaT and using default_ta when needed
                default_ta = pd.Timestamp(ta)
                # we'll use default_ta_time variable later
                default_ta_time = default_ta.time()
                # do not yet fill ta_per_session; we'll compute ta1 = combine(st.date(), default_ta_time) per session
                # indicate by setting a flag
                default_ta_provided = True
            # if execution reaches here with sequence-like, default_ta_provided might be undefined; handle after
        except Exception:
            # fallback: treat as single datetime
            default_ta = pd.Timestamp(ta)
            default_ta_time = default_ta.time()
            default_ta_provided = True

    # Precompute available steps & base Ch_all
    available_steps = np.zeros(n)
    for i in range(n):
        mins = minutes_between_safe(charging_profiles['st'][i], charging_profiles['et'][i])
        available_steps[i] = mins / dt if dt > 0 else 0.0

    Ch_all = np.zeros(n)
    for i in range(n):
        steps = available_steps[i]
        d_val = charging_profiles['d'][i]
        Ch_all[i] = d_val * (60.0/dt) / steps if steps > 0 else 0.0

    # truncnorm parameters (same as before)
    mu1, sigma1 = 0.9, 0.1
    mu2, sigma2 = 0.9, 0.1
    mu3, sigma3 = 0.9, 0.1
    mu4, sigma4 = 0.8, 0.1
    a1, b1 = (0 - mu1) / sigma1, (1 - mu1) / sigma1
    a2, b2 = (0 - mu2) / sigma2, (1 - mu2) / sigma2
    a3, b3 = (0 - mu3) / sigma3, (1 - mu3) / sigma3
    a4, b4 = (0 - mu4) / sigma4, (1 - mu4) / sigma4

    # Prepare outputs: ta_per_session (final), is_modified, attack_type
    ta_out = np.full(n, np.datetime64('NaT'), dtype='datetime64[ns]')
    is_modified = np.zeros(n, dtype=bool)
    attack_type = np.array([''] * n, dtype=object)

    # Helper to fetch per-session ta_i (a pandas Timestamp or None)
    def get_ta_for_session(i, st_i):
        """Return pd.Timestamp or None"""
        # priority: explicit ta_per_session entry if not NaT
        if not pd.isna(ta_per_session[i]):
            return pd.Timestamp(ta_per_session[i])
        # else if default_ta_provided (single datetime input), combine its time-of-day with st.date()
        if 'default_ta_provided' in locals() and default_ta_provided:
            # combine default_ta_time with st_i.date()
            return pd.Timestamp(datetime.combine(pd.Timestamp(st_i).date(), default_ta_time))
        # else no ta specified for this session
        return None

    # For each attack type, apply modifications. Keys expected in compromised_indices like 'N1_ta','N2_ta',...
    # We will not change the overall structure: only modify new_profiles fields and set ta_out/is_modified/attack_type.

    # Type-1: N1_ta
    for i in compromised_indices.get('N1_ta', []):
        i = int(i)
        st = pd.Timestamp(charging_profiles['st'][i])
        et = pd.Timestamp(charging_profiles['et'][i])
        ta_i = get_ta_for_session(i, st)
        if ta_i is None:
            # no ta specified for this session -> skip
            continue
        # ensure ta_i is on the same date as st (if user gave full datetime with different date, still allow compare)
        # But original logic used ta1 = combine(st.date(), ta.time()) ; we accept explicit ta_i directly
        # Check whether ta_i falls in session interval [st, et)
        if not (st <= ta_i < et):
            continue
        # apply Type-1: demand manipulation
        Ch = Ch_all[i]
        d_val = float(charging_profiles['d'][i])
        x = truncnorm.rvs(a1, b1, loc=mu1, scale=sigma1)
        # new_d computed based on remaining window (et - ta_i)
        new_d = x*ACRatt * ((et - ta_i).total_seconds()/3600.0) + d_val
        new_profiles['d'][i] = new_d
        is_modified[i] = True
        ta_out[i] = np.datetime64(pd.Timestamp(ta_i))
        attack_type[i] = 'N1'

    # Type-2
    for i in compromised_indices.get('N2_ta', []):
        i = int(i)
        st = pd.Timestamp(charging_profiles['st'][i])
        et = pd.Timestamp(charging_profiles['et'][i])
        ta_i = get_ta_for_session(i, st)
        if ta_i is None: continue
        if not (st <= ta_i < et): continue
        Ch = Ch_all[i]
        d_val = float(charging_profiles['d'][i])
        x = truncnorm.rvs(a2, b2, loc=mu2, scale=sigma2)
        # compute nT as in original (hours)
        # careful with division: follow original formula intent (avoid zero-division)
        denom = (x*ACRatt + Ch)
        if denom <= 0:
            continue
        # nT calculation adapted to avoid mixing minutes/hours incorrectly
        nT = d_val * ((et - ta_i).total_seconds() / 3600.0) / ((et - st).total_seconds() / 3600.0) / denom
        new_profiles['et'][i] = ta_i + timedelta(hours=nT)
        is_modified[i] = True
        ta_out[i] = np.datetime64(pd.Timestamp(ta_i))
        attack_type[i] = 'N2'

    # Type-3
    for i in compromised_indices.get('N3_ta', []):
        i = int(i)
        st = pd.Timestamp(charging_profiles['st'][i])
        et = pd.Timestamp(charging_profiles['et'][i])
        ta_i = get_ta_for_session(i, st)
        if ta_i is None: continue
        if not (st <= ta_i < et): continue
        Ch = Ch_all[i]
        d_val = float(charging_profiles['d'][i])
        x = truncnorm.rvs(a3, b3, loc=mu3, scale=sigma3)
        denom = (x*ACRatt + Ch)
        if denom <= 0:
            continue
        # original had some mixing of seconds; use hours consistently
        nT = d_val * ((et - ta_i).total_seconds()/3600.0) / ((et - st).total_seconds()/3600.0) / denom
        new_profiles['st'][i] = et - timedelta(hours=nT)
        is_modified[i] = True
        ta_out[i] = np.datetime64(pd.Timestamp(ta_i))
        attack_type[i] = 'N3'

    # Type-4
    for i in compromised_indices.get('N4_ta', []):
        i = int(i)
        st = pd.Timestamp(charging_profiles['st'][i])
        et = pd.Timestamp(charging_profiles['et'][i])
        ta_i = get_ta_for_session(i, st)
        if ta_i is None: continue
        if not (st <= ta_i < et): continue
        Ch = Ch_all[i]
        d_val = float(charging_profiles['d'][i])
        x = truncnorm.rvs(a4, b4, loc=mu4, scale=sigma4)
        ACRxatt = x * control_params['Catt'] * control_params['Chav']
        new_d = ACRxatt * ((et - ta_i).total_seconds()/3600.0) + d_val
        new_profiles['d'][i] = new_d
        # compute nT and set new et
        denom = (Ch + ACRatt)
        if denom <= 0:
            continue
        nT = ((et - ta_i).total_seconds()/3600.0) * (Ch + ACRxatt) / denom
        new_profiles['et'][i] = ta_i + timedelta(hours=nT)
        is_modified[i] = True
        ta_out[i] = np.datetime64(pd.Timestamp(ta_i))
        attack_type[i] = 'N4'

    # Type-5
    for i in compromised_indices.get('N5_ta', []):
        i = int(i)
        st = pd.Timestamp(charging_profiles['st'][i])
        et = pd.Timestamp(charging_profiles['et'][i])
        ta_i = get_ta_for_session(i, st)
        if ta_i is None: continue
        if not (st <= ta_i < et): continue
        Ch = Ch_all[i]
        d_val = float(charging_profiles['d'][i])
        x = truncnorm.rvs(a4, b4, loc=mu4, scale=sigma4)
        ACRxatt = x * control_params['Catt'] * control_params['Chav']
        new_d = ACRxatt * ((et - ta_i).total_seconds()/3600.0) + d_val
        new_profiles['d'][i] = new_d
        denom = (Ch + ACRatt)
        if denom <= 0:
            continue
        nT = ((et - ta_i).total_seconds()/3600.0) * (Ch + ACRxatt) / denom
        new_profiles['st'][i] = et - timedelta(hours=nT)
        is_modified[i] = True
        ta_out[i] = np.datetime64(pd.Timestamp(ta_i))
        attack_type[i] = 'N5'

    # Type-6
    for i in compromised_indices.get('N6_ta', []):
        i = int(i)
        st = pd.Timestamp(charging_profiles['st'][i])
        et = pd.Timestamp(charging_profiles['et'][i])
        ta_i = get_ta_for_session(i, st)
        if ta_i is None: continue
        if not (st <= ta_i < et): continue
        Ch = Ch_all[i]
        d_val = float(charging_profiles['d'][i])
        x = truncnorm.rvs(a4, b4, loc=mu4, scale=sigma4)
        ACRxatt = x * control_params['Catt'] * control_params['Chav']
        new_d = ACRxatt * ((et - ta_i).total_seconds()/3600.0) + d_val
        new_profiles['d'][i] = new_d
        denom = (Ch + ACRatt)
        if denom <= 0:
            continue
        nT = ((et - ta_i).total_seconds()/3600.0) * (Ch + ACRxatt) / denom
        delta = abs(((et - ta_i).total_seconds()/3600.0) - nT) * 60.0  # minutes
        st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
        et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
        # the original branching logic preserved
        if ta_i < st_peek and et > et_peek:
            theta = (et - et_peek).total_seconds() / ((et - ta_i) - (et_peek - st_peek)).total_seconds()
            new_profiles['et'][i] = et - timedelta(minutes=(theta*delta))
            new_profiles['st'][i] = ta_i + timedelta(minutes=((1-theta)*delta))
        elif (ta_i >= st_peek and et > et_peek) or (ta_i >= (st_peek + (et_peek - st_peek)/2) and et <= et_peek):
            theta = 1
            new_profiles['et'][i] = et - timedelta(minutes=(theta*delta))
            new_profiles['st'][i] = ta_i
        elif (ta_i < st_peek and et <= et_peek) or (ta_i >= st_peek  and et <= (st_peek + (et_peek - st_peek)/2)):
            theta = 0
            new_profiles['st'][i] = ta_i + timedelta(minutes=((1-theta)*delta))
        elif ta_i >= st_peek and ta_i <= (st_peek + (et_peek - st_peek)/2) and et <= et_peek and et >=  (st_peek + (et_peek - st_peek)/2):
            theta = (et - (st_peek + (et_peek - st_peek)/2)).total_seconds() / (et - ta_i).total_seconds()
            new_profiles['et'][i] = et - timedelta(minutes=(theta*delta))
        else:
            new_profiles['st'][i] = ta_i
        is_modified[i] = True
        ta_out[i] = np.datetime64(pd.Timestamp(ta_i))
        attack_type[i] = 'N6'

    # attach metadata arrays to new_profiles
    new_profiles['is_modified'] = is_modified
    # convert ta_out to pandas timestamps array for user convenience (NaT preserved)
    new_profiles['ta_per_session'] = pd.to_datetime(ta_out)
    new_profiles['attack_type'] = attack_type

    return new_profiles


def build_power_matrix(charging_profiles, DT_MIN):
    """
    Build a power matrix from the charging profiles.
    
    charging_profiles: dict of numpy arrays 'st', 'et', 'd'
    """
    SLOTS_PER_DAY = 24*60 // DT_MIN
    # Build uniform time‐of‐day grid (0, 00:05, 00:10, … 23:55) as timedeltas since midnight
    slot_offsets = [timedelta(minutes=i*DT_MIN) for i in range(SLOTS_PER_DAY)]
    n_sessions = len(charging_profiles['st'])
    power_matrix = np.zeros((n_sessions, SLOTS_PER_DAY))

    for i in range(n_sessions):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        d = charging_profiles['d'][i]
        disc = charging_profiles['disc'][i]
        # actual charging end = min(requestedDeparture, disconnectTime)
        end = min(et, disc)

        # Compute total charging duration in hours
        total_h = (et - st).total_seconds() / 3600
        P = d / total_h if total_h > 0 else 0

        # Compute offsets since midnight for start/end
        st_offset = timedelta(hours=st.hour, minutes=st.minute, seconds=st.second)
        end_offset = timedelta(hours=end.hour, minutes=end.minute, seconds=end.second)

        day0 = st.date()
        dayN = end.date()
        n_days = (dayN - day0).days + 1  # total calendar days covered

        for idx, offset in enumerate(slot_offsets):
            # How many days this session is active during this time‐of‐day slot?
            if n_days == 1:
                # Same calendar day: only if within [st, end)
                active_days = 1 if (st_offset <= offset < end_offset) else 0
            else:
                # First day contribution?
                first = 1 if offset >= st_offset else 0
                # Last day contribution?
                last  = 1 if offset <  end_offset else 0
                # Middle full days (if any)
                middle = max(0, n_days - 2)
                active_days = first + middle + last

            # Total power = P × number of days covering this slot
            power_matrix[i, idx] = P * active_days

    return power_matrix

def build_power_matrix_st_shift(charging_profiles, DT_MIN, original_profiles, st_shift_indices):
    """
    Build a power matrix from the charging profiles.
    
    charging_profiles: dict of numpy arrays 'st', 'et', 'd'
    """
    SLOTS_PER_DAY = 24*60 // DT_MIN
    # Build uniform time‐of‐day grid (0, 00:05, 00:10, … 23:55) as timedeltas since midnight
    slot_offsets = [timedelta(minutes=i*DT_MIN) for i in range(SLOTS_PER_DAY)]
    n_sessions = len(charging_profiles['st'])
    power_matrix = np.zeros((n_sessions, SLOTS_PER_DAY))

    for i in range(n_sessions):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        d = charging_profiles['d'][i]
        disc = charging_profiles['disc'][i]
        # actual charging end = min(requestedDeparture, disconnectTime)
        end = min(et, disc)

        if i in st_shift_indices:
            end_original_st = min(original_profiles['st'][i] + timedelta(minutes=DT_MIN), disc)

            P_original = original_profiles['d'][i] / ((original_profiles['et'][i] - original_profiles['st'][i]).total_seconds() / 3600)
            
            # Compute total charging duration in hours
            total_h = ((et - st).total_seconds())/ 3600
            P = (d - P_original*DT_MIN/60) / total_h if total_h > 0 else 0

            # Compute offsets since midnight for start/end
            st_offset = timedelta(hours=st.hour, minutes=st.minute, seconds=st.second)
            end_offset = timedelta(hours=end.hour, minutes=end.minute, seconds=end.second)
            original_st_offset = timedelta(hours=original_profiles['st'][i].hour, minutes=original_profiles['st'][i].minute, seconds=original_profiles['st'][i].second)
            end_original_st_offset = timedelta(hours=end_original_st.hour, minutes=end_original_st.minute, seconds=end_original_st.second)
            # original_st_plus_dt_offset = timedelta(hours=original_profiles['st'][i].hour, minutes=original_profiles['st'][i].minute+DT_MIN, seconds=original_profiles['st'][i].second)
            # disc_offset = timedelta(hours=charging_profiles['disc'][i].hour, minutes=charging_profiles['disc'][i].minute, seconds=charging_profiles['disc'][i].second)

            day0 = st.date()
            dayN = end.date()
            n_days = (dayN - day0).days + 1  # total calendar days covered

            day0_original = original_profiles['st'][i].date()
            dayN_end_original_st = end_original_st.date()
            n_days_original_st_to_end = (dayN_end_original_st - day0_original).days + 1  # total calendar days covered

            for idx, offset in enumerate(slot_offsets):
                # How many days this session is active during this time‐of‐day slot?
                if n_days == 1:
                    # Same calendar day: only if within [st, end)
                    #active_days = 1 if ((st_offset <= offset < end_offset) or (original_st_offset <= offset < min(original_st_plus_dt_offset, ))) else 0
                    active_days = 1 if (st_offset <= offset < end_offset) else 0
                else:
                    # First day contribution?
                    first = 1 if offset >= st_offset else 0
                    # Last day contribution?
                    last  = 1 if offset <  end_offset else 0
                    # Middle full days (if any)
                    middle = max(0, n_days - 2)
                    active_days = first + middle + last

                # Total power = P × number of days covering this slot
                power_matrix[i, idx] = P * active_days

                if n_days_original_st_to_end == 1:
                    # Same calendar day: only if within [st, end)
                    active_days = 1 if (original_st_offset <= offset < end_original_st_offset) else 0
                else:
                    # First day contribution?
                    first = 1 if offset >= original_st_offset else 0
                    # Last day contribution?
                    last  = 1 if offset <  end_original_st_offset else 0
                    # Middle full days (if any)
                    middle = max(0, n_days_original_st_to_end - 2)
                    active_days = first + middle + last
                power_matrix[i, idx] += P_original * active_days
                #power_matrix[i, idx] = P_original * active_days
        else:
            # Compute total charging duration in hours
            total_h = (et - st).total_seconds() / 3600
            P = d / total_h if total_h > 0 else 0

            # Compute offsets since midnight for start/end
            st_offset = timedelta(hours=st.hour, minutes=st.minute, seconds=st.second)
            end_offset = timedelta(hours=end.hour, minutes=end.minute, seconds=end.second)

            day0 = st.date()
            dayN = end.date()
            n_days = (dayN - day0).days + 1  # total calendar days covered

            for idx, offset in enumerate(slot_offsets):
                # How many days this session is active during this time‐of‐day slot?
                if n_days == 1:
                    # Same calendar day: only if within [st, end)
                    active_days = 1 if (st_offset <= offset < end_offset) else 0
                else:
                    # First day contribution?
                    first = 1 if offset >= st_offset else 0
                    # Last day contribution?
                    last  = 1 if offset <  end_offset else 0
                    # Middle full days (if any)
                    middle = max(0, n_days - 2)
                    active_days = first + middle + last

                # Total power = P × number of days covering this slot
                power_matrix[i, idx] = P * active_days

    return power_matrix

def build_power_matrix_et_shift(charging_profiles, DT_MIN, original_profiles, et_shift_indices):
    """
    Build a power matrix from the charging profiles.
    
    charging_profiles: dict of numpy arrays 'st', 'et', 'd'
    """
    SLOTS_PER_DAY = 24*60 // DT_MIN
    # Build uniform time‐of‐day grid (0, 00:05, 00:10, … 23:55) as timedeltas since midnight
    slot_offsets = [timedelta(minutes=i*DT_MIN) for i in range(SLOTS_PER_DAY)]
    n_sessions = len(charging_profiles['st'])
    power_matrix = np.zeros((n_sessions, SLOTS_PER_DAY))

    for i in range(n_sessions):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        d = charging_profiles['d'][i]
        disc = charging_profiles['disc'][i]
        # actual charging end = min(requestedDeparture, disconnectTime)
        end = min(et, disc)

        if i in et_shift_indices:
            st2 = st + timedelta(minutes=DT_MIN)
            end_original_st = min(st2, disc)

            P_original = original_profiles['d'][i] / ((original_profiles['et'][i] - original_profiles['st'][i]).total_seconds() / 3600)
            
            # Compute total charging duration in hours
            total_h = ((et - st2).total_seconds())/ 3600
            P = (d - P_original*DT_MIN/60) / total_h if total_h > 0 else 0

            # Compute offsets since midnight for start/end
            st2_offset = timedelta(hours=st2.hour, minutes=st2.minute, seconds=st2.second)
            end_offset = timedelta(hours=end.hour, minutes=end.minute, seconds=end.second)
            original_st_offset = timedelta(hours=original_profiles['st'][i].hour, minutes=original_profiles['st'][i].minute, seconds=original_profiles['st'][i].second)
            end_original_st_offset = timedelta(hours=end_original_st.hour, minutes=end_original_st.minute, seconds=end_original_st.second)
            # original_st_plus_dt_offset = timedelta(hours=original_profiles['st'][i].hour, minutes=original_profiles['st'][i].minute+DT_MIN, seconds=original_profiles['st'][i].second)
            # disc_offset = timedelta(hours=charging_profiles['disc'][i].hour, minutes=charging_profiles['disc'][i].minute, seconds=charging_profiles['disc'][i].second)

            day0 = st2.date()
            dayN = end.date()
            n_days = (dayN - day0).days + 1  # total calendar days covered

            day0_original = original_profiles['st'][i].date()
            dayN_end_original_st = end_original_st.date()
            n_days_original_st_to_end = (dayN_end_original_st - day0_original).days + 1  # total calendar days covered

            for idx, offset in enumerate(slot_offsets):
                # How many days this session is active during this time‐of‐day slot?
                if n_days == 1:
                    # Same calendar day: only if within [st, end)
                    active_days = 1 if (st2_offset <= offset < end_offset) else 0
                    #active_days = 1 if ((st_offset <= offset < end_offset) or (original_st_offset <= offset < min(original_st_plus_dt_offset, ))) else 0
                    # if (st_offset + timedelta(minutes=DT_MIN) <= offset < end_offset):
                    #     active_days = 1
                    #     power_matrix[i, idx] = P * active_days
                    # elif (st_offset <= offset < min(st_offset + timedelta(minutes=DT_MIN), end_offset)):
                    #     active_days = 1
                    #     power_matrix[i, idx] = P_original * active_days
                else:
                    # First day contribution?
                    first = 1 if offset >= st2_offset else 0
                    # Last day contribution?
                    last  = 1 if offset <  end_offset else 0
                    # Middle full days (if any)
                    middle = max(0, n_days - 2)
                    active_days = first + middle + last

                # Total power = P × number of days covering this slot
                power_matrix[i, idx] = P * active_days

                if n_days_original_st_to_end == 1:
                    # Same calendar day: only if within [st, end)
                    active_days = 1 if (original_st_offset <= offset < end_original_st_offset) else 0
                else:
                    # First day contribution?
                    first = 1 if offset >= original_st_offset else 0
                    # Last day contribution?
                    last  = 1 if offset <  end_original_st_offset else 0
                    # Middle full days (if any)
                    middle = max(0, n_days_original_st_to_end - 2)
                    active_days = first + middle + last
                power_matrix[i, idx] += P_original * active_days
        else:
            # Compute total charging duration in hours
            total_h = (et - st).total_seconds() / 3600
            P = d / total_h if total_h > 0 else 0

            # Compute offsets since midnight for start/end
            st_offset = timedelta(hours=st.hour, minutes=st.minute, seconds=st.second)
            end_offset = timedelta(hours=end.hour, minutes=end.minute, seconds=end.second)

            day0 = st.date()
            dayN = end.date()
            n_days = (dayN - day0).days + 1  # total calendar days covered

            for idx, offset in enumerate(slot_offsets):
                # How many days this session is active during this time‐of‐day slot?
                if n_days == 1:
                    # Same calendar day: only if within [st, end)
                    active_days = 1 if (st_offset <= offset < end_offset) else 0
                else:
                    # First day contribution?
                    first = 1 if offset >= st_offset else 0
                    # Last day contribution?
                    last  = 1 if offset <  end_offset else 0
                    # Middle full days (if any)
                    middle = max(0, n_days - 2)
                    active_days = first + middle + last

                # Total power = P × number of days covering this slot
                power_matrix[i, idx] = P * active_days

    return power_matrix

def build_power_matrix_ta(New_profiles, DT_MIN, ta, original_profiles, ta_indices):
    """
    Build a power matrix from the charging profiles.
    
    New_profiles: dict of numpy arrays 'st', 'et', 'd'
    """
    SLOTS_PER_DAY = 24*60 // DT_MIN
    # Build uniform time‐of‐day grid (0, 00:05, 00:10, … 23:55) as timedeltas since midnight
    slot_offsets = [timedelta(minutes=i*DT_MIN) for i in range(SLOTS_PER_DAY)]
    n_sessions = len(New_profiles['st'])
    power_matrix = np.zeros((n_sessions, SLOTS_PER_DAY))

    for i in range(n_sessions):
        st = New_profiles['st'][i]
        et = New_profiles['et'][i]
        d = New_profiles['d'][i]
        disc = New_profiles['disc'][i]
        # actual charging end = min(requestedDeparture, disconnectTime)
        end = min(et, disc)

        if i in ta_indices:
            st_ori = original_profiles['st'][i]
            et_ori = original_profiles['et'][i]
            ta = datetime.combine(st.date(), ta.time())
            end_original_st = min(ta, disc)

            P_original = original_profiles['d'][i] / ((et_ori - st_ori).total_seconds() / 3600)
            
            # Compute total charging duration in hours
            if st == st_ori:
                total_h = ((et - ta).total_seconds())/ 3600
                st_new_p = ta
            else:
                total_h = ((et - st).total_seconds())/ 3600
                st_new_p = st
            
            ori_p_time_min = (ta - st_ori).total_seconds()/60
            P = (d - P_original*ori_p_time_min/60) / total_h if total_h > 0 else 0

            # Compute offsets since midnight for start/end
            st_offset = timedelta(hours=st_new_p.hour, minutes=st_new_p.minute, seconds=st_new_p.second)
            end_offset = timedelta(hours=end.hour, minutes=end.minute, seconds=end.second)
            original_st_offset = timedelta(hours=original_profiles['st'][i].hour, minutes=original_profiles['st'][i].minute, seconds=original_profiles['st'][i].second)
            end_original_st_offset = timedelta(hours=end_original_st.hour, minutes=end_original_st.minute, seconds=end_original_st.second)
            # original_st_plus_dt_offset = timedelta(hours=original_profiles['st'][i].hour, minutes=original_profiles['st'][i].minute+DT_MIN, seconds=original_profiles['st'][i].second)
            # disc_offset = timedelta(hours=New_profiles['disc'][i].hour, minutes=New_profiles['disc'][i].minute, seconds=New_profiles['disc'][i].second)

            day0 = st_new_p.date()
            dayN = end.date()
            n_days = (dayN - day0).days + 1  # total calendar days covered

            day0_original = original_profiles['st'][i].date()
            dayN_end_original_st = end_original_st.date()
            n_days_original_st_to_end = (dayN_end_original_st - day0_original).days + 1  # total calendar days covered

            for idx, offset in enumerate(slot_offsets):
                # How many days this session is active during this time‐of‐day slot?
                if n_days == 1:
                    # Same calendar day: only if within [st, end)
                    #active_days = 1 if ((st_offset <= offset < end_offset) or (original_st_offset <= offset < min(original_st_plus_dt_offset, ))) else 0
                    active_days = 1 if (st_offset <= offset < end_offset) else 0
                    power_matrix[i, idx] = P * active_days
                elif n_days > 1:
                    # First day contribution?
                    first = 1 if offset >= st_offset else 0
                    # Last day contribution?
                    last  = 1 if offset < end_offset else 0
                    # Middle full days (if any)
                    middle = max(0, n_days - 2)
                    active_days = first + middle + last
                    # Total power = P × number of days covering this slot
                    power_matrix[i, idx] = P * active_days

                if n_days_original_st_to_end == 1:
                    # Same calendar day: only if within [st, end)
                    active_days = 1 if (original_st_offset <= offset < end_original_st_offset) else 0
                    power_matrix[i, idx] += P_original * active_days
                elif n_days_original_st_to_end > 1:
                    # First day contribution?
                    first = 1 if offset >= original_st_offset else 0
                    # Last day contribution?
                    last  = 1 if offset <  end_original_st_offset else 0
                    # Middle full days (if any)
                    middle = max(0, n_days_original_st_to_end - 2)
                    active_days = first + middle + last
                    power_matrix[i, idx] += P_original * active_days
                #power_matrix[i, idx] = P_original * active_days
        else:
            # Compute total charging duration in hours
            total_h = (et - st).total_seconds() / 3600
            P = d / total_h if total_h > 0 else 0

            # Compute offsets since midnight for start/end
            st_offset = timedelta(hours=st.hour, minutes=st.minute, seconds=st.second)
            end_offset = timedelta(hours=end.hour, minutes=end.minute, seconds=end.second)

            day0 = st.date()
            dayN = end.date()
            n_days = (dayN - day0).days + 1  # total calendar days covered

            for idx, offset in enumerate(slot_offsets):
                # How many days this session is active during this time‐of‐day slot?
                if n_days == 1:
                    # Same calendar day: only if within [st, end)
                    active_days = 1 if (st_offset <= offset < end_offset) else 0
                else:
                    # First day contribution?
                    first = 1 if offset >= st_offset else 0
                    # Last day contribution?
                    last  = 1 if offset <  end_offset else 0
                    # Middle full days (if any)
                    middle = max(0, n_days - 2)
                    active_days = first + middle + last

                # Total power = P × number of days covering this slot
                power_matrix[i, idx] = P * active_days

    return power_matrix


def plot_power_matrix(power_matrix, DT_MIN):
    """
    Plot the power matrix as a time‐of‐day load profile.
    
    power_matrix: numpy array of shape (n_sessions, SLOTS_PER_DAY)
    """
    agg_power = power_matrix.sum(axis=0)

    # Build an x‐axis in hours since midnight
    slots_per_day = len(agg_power)
    x_hours = [(i * DT_MIN) / 60 for i in range(slots_per_day)]

    # Plot
    plt.figure(figsize=(10, 4))
    plt.plot(x_hours, agg_power)
    plt.xlabel("Time of day (hours since midnight)")
    plt.ylabel("Total charging power (kW)")
    plt.title("Aggregate Charging Load by Time‐of‐Day Slot")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_two_power_matrices(pm1, pm2, DT_MIN, labels=('First','Second')):
    """
    Plot two aggregate‐load profiles on the same figure.

    pm1, pm2: numpy arrays of shape (n_sessions, SLOTS_PER_DAY)
    DT_MIN:   slot length in minutes
    labels:   tuple of two strings, legend labels for pm1 and pm2
    """
    # Aggregate each
    agg1 = pm1.sum(axis=0)
    agg2 = pm2.sum(axis=0)

    # X axis in hours since midnight
    slots = len(agg1)
    x = [(i * DT_MIN) / 60 for i in range(slots)]

    # Plot both
    plt.figure(figsize=(10,4))
    plt.plot(x, agg1, label=labels[0])
    plt.plot(x, agg2, label=labels[1], linestyle='--')
    plt.xlabel("Time of day (hours since midnight)")
    plt.ylabel("Total charging power (kW)")
    plt.title("Aggregate Charging Load by Time‐of‐Day Slot")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_multiple_power_matrices(original_power_matrix, attacked_power_matrices, DT_MIN, labels):
    # plot different power matrices with different colors
    agg_original = original_power_matrix.sum(axis=0)
    slots = len(agg_original)
    x = [(i * DT_MIN) / 60 for i in range(slots)]
    plt.figure(figsize=(10,4))
    plt.plot(x, agg_original, label=labels[0], color='black', linewidth=2)
    colors = plt.cm.viridis(np.linspace(0, 1, len(attacked_power_matrices)))
    for i, pm in enumerate(attacked_power_matrices):
        agg = pm.sum(axis=0)
        plt.plot(x, agg, label=labels[i+1], color=colors[i], alpha=0.7)
    plt.xlabel("Time of day (hours since midnight)")
    plt.ylabel("Total charging power (kW)")
    plt.title("Aggregate Charging Load by Time‐of‐Day Slot")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_6_power_matrices(original_power_matrix, attacked_power_matrices, DT_MIN, labels):
    # plot different power matrices with different line styles and colors
    agg_original = original_power_matrix.sum(axis=0)
    slots = len(agg_original)
    x = [(i * DT_MIN) / 60 for i in range(slots)]
    plt.figure(figsize=(10,4))
    plt.plot(x, agg_original, label=labels[0], color='black', linewidth=2)
    linestyles = ['-', '--', '-.', ':', (0, (3, 1, 1)), (0, (5, 1))]
    colors = plt.cm.viridis(np.linspace(0, 1, len(attacked_power_matrices)))
    for i, pm in enumerate(attacked_power_matrices):
        agg = pm.sum(axis=0)
        plt.plot(x, agg, label=labels[i+1], color=colors[i], linestyle=linestyles[i % len(linestyles)], alpha=0.7)
    plt.xlabel("Time of day (hours since midnight)")
    plt.ylabel("Total charging power (kW)")
    plt.title("Aggregate Charging Load by Time‐of‐Day Slot")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

def select_80_percent(original_list):
    n = int(len(original_list) * 0.8)
    return random.sample(original_list, n)



In [ ]:
# for SEED in range(1, 1000):
# SEED = 1
# os.environ['PYTHONHASHSEED'] = str(SEED)
# random.seed(SEED)
# np.random.seed(SEED)
# control_params = { "Catt":0.08, "Chav":30, "Pl":24, "dt":5 }
# DT_MIN = control_params['dt']    # slot length in minutes
# #SLOTS_PER_DAY = 24*60 // DT_MIN
# # Load JSON, build charging_profiles as before...
# with open("caltech_sessions_2021.json") as f:
#     data = json.load(f)

# st_list, et_list, d_list, disc_list = [], [], [], []

# for rec in data:
#     ui_list = rec.get("userInputs")
#     if not ui_list:
#         # ui_list 为 None 或空列表时跳过
#         continue
#     ui = ui_list[0]
#     if float(ui["kWhRequested"]) <= 0:
#         # kWhRequested 为 0 或负数时跳过
#         continue
#     # original modifiedAt → local start
#     st_list.append(parse_datetime(ui["modifiedAt"]))
#     # requestedDeparture → local end
#     et_list.append(parse_datetime(ui["requestedDeparture"]))
#     # energy requested
#     d_list.append(ui["kWhRequested"])
#     # parse disconnectTime (ISO 8601 with -08:00 offset) and drop TZinfo → naive local dt
#     dt_disc = datetime.fromisoformat(rec["disconnectTime"])
#     # .replace(tzinfo=None) makes it a naive datetime in local wall‐clock time
#     disc_list.append(dt_disc.replace(tzinfo=None))

# charging_profiles = {
#     "st": np.array(st_list),
#     "et": np.array(et_list),
#     "d":  np.array(d_list),
#     "disc": np.array(disc_list),
# }
# import pandas as pd
# #Create a meaningful list of indices for each type of attack
# Nhd, Nhe, Nhs, N4, N5, N6 = [], [], [], [], [], []
# N1_ta, N2_ta, N3_ta, N4_ta, N5_ta, N6_ta = [], [], [], [], [], []
# ta = datetime(2025, 9, 4, 18, 55, 00)
# print(ta)

# for i in range(len(d_list)):
#     st = charging_profiles['st'][i]
#     et = charging_profiles['et'][i]
#     ta_dt = st + timedelta(minutes = DT_MIN)
#     st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
#     et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
#     if ta_dt < et_peek and et > st_peek:
#         Nhd.append(i)
#     if ta_dt < et_peek and et > et_peek:
#         Nhe.append(i)
#     if ta_dt < st_peek and et > st_peek:
#         Nhs.append(i)
#     if ta_dt < et_peek and et > st_peek:
#         N4.append(i)
#     if ta_dt < et_peek and et > st_peek:
#         N5.append(i)
#     if ta_dt < et_peek and et > st_peek:
#         N6.append(i)

#     ta1 = datetime.combine(st.date(), ta.time())
#     if st <= ta1 < et:
#         ta = ta1
#         if ta < et_peek and et > st_peek:
#             N1_ta.append(i)
#         if ta < st_peek and et > st_peek:
#             N2_ta.append(i)
#         if ta < et_peek and et > et_peek:
#             N3_ta.append(i)
#         if ta < et_peek and et > st_peek:
#             N4_ta.append(i)
#         if ta < et_peek and et > st_peek:
#             N5_ta.append(i)
#         if ta < et_peek and et > st_peek:
#             N6_ta.append(i)
    

# # Print the meaningful indices for each type of attack
# print("Meaningful indices for each type of attack:")
# # print("Nhd (Type-1):", Nhd)
# # print("Nhe (Type-2):", Nhe)
# # print("Nhs (Type-3):", Nhs)
# # print("N4 (Type-4):", N4)
# # print("N5 (Type-5):", N5)
# print("N6 (Type-6):", N6)
# print(len(N6))

# Nhd_indices = {"Nhd": Nhd} # Type‑1
# Nhe_indices = {"Nhe": Nhe} # Type‑2
# Nhs_indices = {"Nhs": Nhs} # Type‑3
# N4_indices = {"N4": N4} # Type‑4
# N5_indices = {"N5": N5} # Type‑5
# N6_indices = {"N6": N6} # Type‑6

# print("N1_ta (Type-1):", N1_ta)
# print("N2_ta (Type-2):", N2_ta)
# print("N3_ta (Type-3):", N3_ta)
# print("N4_ta (Type-4):", N4_ta)
# print("N5_ta (Type-5):", N5_ta)
# print("N6_ta (Type-6):", N6_ta)

# N1_ta_indices = {"N1_ta": N1_ta} # Type‑1
# N2_ta_indices = {"N2_ta": N2_ta} # Type‑2
# N3_ta_indices = {"N3_ta": N3_ta} # Type‑3
# N4_ta_indices = {"N4_ta": N4_ta} # Type‑4
# N5_ta_indices = {"N5_ta": N5_ta} # Type‑5
# N6_ta_indices = {"N6_ta": N6_ta} # Type‑6

# #Type-1-ta
# N1_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N1_ta_indices, ta)
# #plot_power_matrix(build_power_matrix(Nhd_profiles, DT_MIN), DT_MIN)

# #Type-2-ta
# ta = datetime(2025, 9, 4, 15, 55, 00)
# N2_ta = []
# for i in range(len(d_list)):
#     st = charging_profiles['st'][i]
#     et = charging_profiles['et'][i]
#     st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
#     et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
#     ta1 = datetime.combine(st.date(), ta.time())
#     if st <= ta1 < et:
#         ta = ta1
#         if ta < st_peek and et > st_peek:
#             N2_ta.append(i)

# N2_ta_indices = {"N2_ta": N2_ta} # Type‑2
# N2_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N2_ta_indices, ta)

# #Type-3-ta
# ta = datetime(2025, 9, 4, 15, 55, 00)
# N3_ta = []
# for i in range(len(d_list)):
#     st = charging_profiles['st'][i]
#     et = charging_profiles['et'][i]
#     st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
#     et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
#     ta1 = datetime.combine(st.date(), ta.time())
#     if st <= ta1 < et:
#         ta = ta1
#         if ta < et_peek and et > et_peek:
#             N3_ta.append(i)
            
# N3_ta_indices = {"N3_ta": N3_ta} # Type‑3
# N3_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N3_ta_indices, ta)

# #Type-4-ta
# ta = datetime(2025, 9, 4, 15, 55, 00)
# N4_ta = []
# for i in range(len(d_list)):
#     st = charging_profiles['st'][i]
#     et = charging_profiles['et'][i]
#     st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
#     et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
#     ta1 = datetime.combine(st.date(), ta.time())
#     if st <= ta1 < et:
#         ta = ta1
#         if ta < et_peek and et > st_peek:
#             N4_ta.append(i)

# N4_ta_indices = {"N4_ta": N4_ta} # Type‑4
# N4_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N4_ta_indices, ta)

# #Type-5-ta
# ta = datetime(2025, 9, 5, 15, 55, 00)
# N5_ta = []
# for i in range(len(d_list)):
#     st = charging_profiles['st'][i]
#     et = charging_profiles['et'][i]
#     st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
#     et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
#     ta1 = datetime.combine(st.date(), ta.time())
#     if st <= ta1 < et:
#         ta = ta1
#         if ta < et_peek and et > st_peek:
#             N5_ta.append(i)

# N5_ta_indices = {"N5_ta": N5_ta} # Type‑5
# N5_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N5_ta_indices, ta)





In [ ]:
SEED = 4
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
control_params = { "Catt":0.15, "Chav":30, "Pl":24, "dt":5 }
DT_MIN = control_params['dt']    # slot length in minutes
#SLOTS_PER_DAY = 24*60 // DT_MIN
# Load JSON, build charging_profiles as before...
with open("caltech_sessions_2021.json") as f:
    data = json.load(f)

st_list, et_list, d_list, disc_list = [], [], [], []

for rec in data:
    ui_list = rec.get("userInputs")
    if not ui_list:
        # ui_list 为 None 或空列表时跳过
        continue
    ui = ui_list[0]
    if float(ui["kWhRequested"]) <= 0:
        # kWhRequested 为 0 或负数时跳过
        continue
    # original modifiedAt → local start
    st_list.append(parse_datetime(ui["modifiedAt"]))
    # requestedDeparture → local end
    et_list.append(parse_datetime(ui["requestedDeparture"]))
    # energy requested
    d_list.append(ui["kWhRequested"])
    # parse disconnectTime (ISO 8601 with -08:00 offset) and drop TZinfo → naive local dt
    dt_disc = datetime.fromisoformat(rec["disconnectTime"])
    # .replace(tzinfo=None) makes it a naive datetime in local wall‐clock time
    disc_list.append(dt_disc.replace(tzinfo=None))

charging_profiles = {
    "st": np.array(st_list),
    "et": np.array(et_list),
    "d":  np.array(d_list),
    "disc": np.array(disc_list),
}




In [ ]:
import pandas as pd
#Create a meaningful list of indices for each type of attack
Nhd, Nhe, Nhs, N4, N5, N6 = [], [], [], [], [], []
N1_ta, N2_ta, N3_ta, N4_ta, N5_ta, N6_ta = [], [], [], [], [], []
ta = datetime(2025, 9, 4, 18, 55, 00)
print(ta)

for i in range(len(d_list)):
    st = charging_profiles['st'][i]
    et = charging_profiles['et'][i]
    ta_dt = st + timedelta(minutes = DT_MIN)
    st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
    et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
    if ta_dt < et_peek and et > st_peek:
        Nhd.append(i)
    if ta_dt < et_peek and et > et_peek:
        Nhe.append(i)
    if ta_dt < st_peek and et > st_peek:
        Nhs.append(i)
    if ta_dt < et_peek and et > st_peek:
        N4.append(i)
    if ta_dt < et_peek and et > st_peek:
        N5.append(i)
    if ta_dt < et_peek and et > st_peek:
        N6.append(i)

    ta1 = datetime.combine(st.date(), ta.time())
    if st <= ta1 < et:
        ta = ta1
        if ta < et_peek and et > st_peek:
            N1_ta.append(i)
        if ta < st_peek and et > st_peek:
            N2_ta.append(i)
        if ta < et_peek and et > et_peek:
            N3_ta.append(i)
        if ta < et_peek and et > st_peek:
            N4_ta.append(i)
        if ta < et_peek and et > st_peek:
            N5_ta.append(i)
        if ta < et_peek and et > st_peek:
            N6_ta.append(i)
    

# Print the meaningful indices for each type of attack
print("Meaningful indices for each type of attack:")
# print("Nhd (Type-1):", Nhd)
# print("Nhe (Type-2):", Nhe)
# print("Nhs (Type-3):", Nhs)
# print("N4 (Type-4):", N4)
# print("N5 (Type-5):", N5)
print("N6 (Type-6):", N6)
print(len(N6))

Nhd_indices = {"Nhd": Nhd} # Type‑1
Nhe_indices = {"Nhe": Nhe} # Type‑2
Nhs_indices = {"Nhs": Nhs} # Type‑3
N4_indices = {"N4": N4} # Type‑4
N5_indices = {"N5": N5} # Type‑5
N6_indices = {"N6": N6} # Type‑6

print("N1_ta (Type-1):", N1_ta)
print("N2_ta (Type-2):", N2_ta)
print("N3_ta (Type-3):", N3_ta)
print("N4_ta (Type-4):", N4_ta)
print("N5_ta (Type-5):", N5_ta)
print("N6_ta (Type-6):", N6_ta)

N1_ta_indices = {"N1_ta": N1_ta} # Type‑1
N2_ta_indices = {"N2_ta": N2_ta} # Type‑2
N3_ta_indices = {"N3_ta": N3_ta} # Type‑3
N4_ta_indices = {"N4_ta": N4_ta} # Type‑4
N5_ta_indices = {"N5_ta": N5_ta} # Type‑5
N6_ta_indices = {"N6_ta": N6_ta} # Type‑6






In [ ]:
#Type-1-ta
N1_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N1_ta_indices, ta)
#plot_power_matrix(build_power_matrix(Nhd_profiles, DT_MIN), DT_MIN)

In [ ]:
#Type-2-ta
ta = datetime(2025, 9, 4, 15, 55, 00)
N2_ta = []
for i in range(len(d_list)):
    st = charging_profiles['st'][i]
    et = charging_profiles['et'][i]
    st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
    et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
    ta1 = datetime.combine(st.date(), ta.time())
    if st <= ta1 < et:
        ta = ta1
        if ta < st_peek and et > st_peek:
            N2_ta.append(i)

N2_ta_indices = {"N2_ta": N2_ta} # Type‑2
N2_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N2_ta_indices, ta)

In [ ]:
#Type-3-ta
ta = datetime(2025, 9, 4, 15, 55, 00)
N3_ta = []
for i in range(len(d_list)):
    st = charging_profiles['st'][i]
    et = charging_profiles['et'][i]
    st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
    et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
    ta1 = datetime.combine(st.date(), ta.time())
    if st <= ta1 < et:
        ta = ta1
        if ta < et_peek and et > et_peek:
            N3_ta.append(i)
            
N3_ta_indices = {"N3_ta": N3_ta} # Type‑3
N3_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N3_ta_indices, ta)

In [ ]:
#Type-4-ta
ta = datetime(2025, 9, 4, 15, 55, 00)
N4_ta = []
for i in range(len(d_list)):
    st = charging_profiles['st'][i]
    et = charging_profiles['et'][i]
    st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
    et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
    ta1 = datetime.combine(st.date(), ta.time())
    if st <= ta1 < et:
        ta = ta1
        if ta < et_peek and et > st_peek:
            N4_ta.append(i)

N4_ta_indices = {"N4_ta": N4_ta} # Type‑4
N4_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N4_ta_indices, ta)

In [ ]:
#Type-5-ta
ta = datetime(2025, 9, 5, 15, 55, 00)
N5_ta = []
for i in range(len(d_list)):
    st = charging_profiles['st'][i]
    et = charging_profiles['et'][i]
    st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
    et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
    ta1 = datetime.combine(st.date(), ta.time())
    if st <= ta1 < et:
        ta = ta1
        if ta < et_peek and et > st_peek:
            N5_ta.append(i)

N5_ta_indices = {"N5_ta": N5_ta} # Type‑5
N5_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N5_ta_indices, ta)

In [ ]:
#Type-6-ta
ta = datetime(2026, 9, 4, 18, 55, 00)
N6_ta = []
for i in range(len(d_list)):
    st = charging_profiles['st'][i]
    et = charging_profiles['et'][i]
    st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
    et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
    ta1 = datetime.combine(st.date(), ta.time())
    if st <= ta1 < et:
        ta = ta1
        if ta < et_peek and et > st_peek:
            N6_ta.append(i)

N6_ta_indices = {"N6_ta": N6_ta} # Type‑6
N6_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N6_ta_indices, ta)

In [ ]:
#Type-1
Nhd_profiles = launch_CMAs(charging_profiles, control_params, Nhd_indices)

In [ ]:
#Type-2
Nhe_profiles = launch_CMAs(charging_profiles, control_params, Nhe_indices)

In [ ]:
#Type-3
Nhs_profiles = launch_CMAs(charging_profiles, control_params, Nhs_indices)

In [ ]:
#Type-4
N4_profiles = launch_CMAs(charging_profiles, control_params, N4_indices)

In [ ]:
#Type_5
N5_profiles = launch_CMAs(charging_profiles, control_params, N5_indices)

In [ ]:
#Type-6
N6_indices_select = {"N6": N6} # Type‑6
N6_profiles = launch_CMAs(charging_profiles, control_params, N6_indices_select)

In [ ]:
# Three-way split feature preparation
import numpy as np

THREE_SPLIT_SEED = 66
TRAIN_FRACTION = 0.60
VALIDATION_FRACTION = 0.20
TEST_FRACTION = 0.20


def generate_features(charging_profiles, DT_MIN, N6_select):
    """
    Generate the 5 session-level features used by AE + RealNVP.

    Features:
    - requested energy d
    - requested energy per connected hour
    - start-time offset from peak-window midpoint
    - requested-departure offset from peak-window midpoint
    - connected duration in hours
    """
    features = []
    labels = []
    selected = set(map(int, N6_select))

    for i in range(len(charging_profiles['st'])):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        d = charging_profiles['d'][i]

        st_peak = datetime(st.year, st.month, st.day, 16, 0, 0)
        et_peak = datetime(st.year, st.month, st.day, 21, 0, 0)
        mid_peak = st_peak + (et_peak - st_peak) / 2

        duration = (et - st).total_seconds() / 3600
        d_per_hour = d / duration if duration > 0 else 0
        st_offset = (st - mid_peak).total_seconds()
        et_offset = (et - mid_peak).total_seconds()

        features.append([d, d_per_hour, st_offset, et_offset, duration])
        labels.append(1 if i in selected else 0)

    return np.array(features, dtype=float), np.array(labels, dtype=int)


def deterministic_three_way_split(indices, train_frac=TRAIN_FRACTION, val_frac=VALIDATION_FRACTION, seed=THREE_SPLIT_SEED):
    idx = np.array(sorted(map(int, indices)), dtype=int)
    rng = np.random.default_rng(int(seed))
    rng.shuffle(idx)
    n = len(idx)
    n_train = int(round(n * train_frac))
    n_val = int(round(n * val_frac))
    n_train = min(max(n_train, 0), n)
    n_val = min(max(n_val, 0), n - n_train)
    train = sorted(idx[:n_train].astype(int).tolist())
    validation = sorted(idx[n_train:n_train + n_val].astype(int).tolist())
    test = sorted(idx[n_train + n_val:].astype(int).tolist())
    return train, validation, test


def build_three_way_session_split(n_sessions, attack_reference_indices, seed=THREE_SPLIT_SEED):
    all_indices = list(range(int(n_sessions)))
    attack_reference = sorted(set(map(int, attack_reference_indices)))
    non_reference = [i for i in all_indices if i not in set(attack_reference)]

    ref_train, ref_validation, ref_test = deterministic_three_way_split(attack_reference, seed=seed)
    non_train, non_validation, non_test = deterministic_three_way_split(non_reference, seed=seed + 1009)

    split = {
        'train': sorted(non_train + ref_train),
        'validation': sorted(non_validation + ref_validation),
        'test': sorted(non_test + ref_test),
        'attack_reference_train': ref_train,
        'attack_reference_validation': ref_validation,
        'attack_reference_test': ref_test,
    }
    sets = {k: set(v) for k, v in split.items() if k in ['train', 'validation', 'test']}
    leakage = {
        'train_validation_overlap': len(sets['train'] & sets['validation']),
        'train_test_overlap': len(sets['train'] & sets['test']),
        'validation_test_overlap': len(sets['validation'] & sets['test']),
        'unassigned_sessions': n_sessions - len(sets['train'] | sets['validation'] | sets['test']),
    }
    if any(leakage.values()):
        raise RuntimeError(f'Three-way split leakage detected: {leakage}')
    split['leakage'] = leakage
    return split


def subset_indices(split_indices, candidate_indices):
    split_set = set(map(int, split_indices))
    return sorted(i for i in map(int, candidate_indices) if i in split_set)


three_way_split = build_three_way_session_split(len(d_list), N6, seed=THREE_SPLIT_SEED)
print('[three-way split]', {k: len(three_way_split[k]) for k in ['train', 'validation', 'test']})
print('[attack reference split]', {
    'train': len(three_way_split['attack_reference_train']),
    'validation': len(three_way_split['attack_reference_validation']),
    'test': len(three_way_split['attack_reference_test']),
})
print('[leakage check]', three_way_split['leakage'])

charging_features, _ = generate_features(charging_profiles, DT_MIN, [])

attack_profile_specs = {
    'Type-1': (Nhd_profiles, Nhd),
    'Type-2': (Nhe_profiles, Nhe),
    'Type-3': (Nhs_profiles, Nhs),
    'Type-4': (N4_profiles, N4),
    'Type-5': (N5_profiles, N5),
    'Type-6': (N6_profiles, N6),
    'Type-1-ta': (N1_ta_profiles, N1_ta),
    'Type-2-ta': (N2_ta_profiles, N2_ta),
    'Type-3-ta': (N3_ta_profiles, N3_ta),
    'Type-4-ta': (N4_ta_profiles, N4_ta),
    'Type-5-ta': (N5_ta_profiles, N5_ta),
    'Type-6-ta': (N6_ta_profiles, N6_ta),
}

validation_attack_counts = {}
test_attack_counts = {}
for name, (_, indices_all) in attack_profile_specs.items():
    validation_attack_counts[name] = len(subset_indices(three_way_split['validation'], indices_all))
    test_attack_counts[name] = len(subset_indices(three_way_split['test'], indices_all))

print('[validation attack counts]', validation_attack_counts)
print('[held-out test attack counts]', test_attack_counts)
print('Training clean feature shape:', charging_features[three_way_split['train'], 0:5].shape)
print('Validation clean feature shape:', charging_features[three_way_split['validation'], 0:5].shape)
print('Held-out test clean feature shape:', charging_features[three_way_split['test'], 0:5].shape)


In [ ]:
# paste this single cell into your notebook and run it
import os
import random
import numpy as np
import torch
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from datetime import datetime
import torch.nn as nn
import shutil
import json


# -----------------------
# 配置（可按需调整）
# -----------------------
BEST_SEED = 66  # 已找到的最优种子（单线程运行）

# =========================
# Hyperparameters & Device
# =========================
latent_dim    = 3
batch_size    = 32
num_epochs    = 1000
learning_rate = 1e-3
alpha         = 0.05

# =========================
# Normalizing Flow (RealNVP) hyperparameters (only for detection head)
# =========================
FLOW_HIDDEN   = 128
FLOW_LAYERS   = 12
FLOW_EPOCHS   = 160     # tuned for stronger residual-density modeling
FLOW_LR       = 1e-3
FLOW_BS       = 256

# =========================
# Define Autoencoder Model
# =========================
class Autoencoder(nn.Module):
    def __init__(self, input_dim=5, latent_dim=latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(True),
            nn.Linear(32, 16),
            nn.ReLU(True),
            nn.Linear(16, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(True),
            nn.Linear(16, 32),
            nn.ReLU(True),
            nn.Linear(32, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# =========================
# Normalizing Flow (RealNVP) for error-space detection head
# =========================
class AffineCoupling(nn.Module):
    def __init__(self, dim, mask, hidden=64):
        super().__init__()
        self.dim = dim
        self.register_buffer("mask", mask)  # shape (dim,)
        # s,t nets take masked part as input, output full dim then we apply (1-mask)
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(True),
            nn.Linear(hidden, hidden),
            nn.ReLU(True),
            nn.Linear(hidden, dim * 2),
        )

    def forward(self, x):
        # x: (B, dim)
        x_masked = x * self.mask
        st = self.net(x_masked)
        s, t = torch.chunk(st, 2, dim=-1)
        # stabilize scale
        s = torch.tanh(s)
        # apply only to unmasked dims
        s = s * (1.0 - self.mask)
        t = t * (1.0 - self.mask)
        y = x_masked + (1.0 - self.mask) * (x * torch.exp(s) + t)
        logdet = s.sum(dim=-1)  # (B,)
        return y, logdet

    def inverse(self, y):
        y_masked = y * self.mask
        st = self.net(y_masked)
        s, t = torch.chunk(st, 2, dim=-1)
        s = torch.tanh(s)
        s = s * (1.0 - self.mask)
        t = t * (1.0 - self.mask)
        x = y_masked + (1.0 - self.mask) * ((y - t) * torch.exp(-s))
        logdet = (-s).sum(dim=-1)
        return x, logdet

class RealNVP(nn.Module):
    def __init__(self, dim, n_layers=6, hidden=64):
        super().__init__()
        self.dim = dim
        self.n_layers = n_layers
        masks = []
        # alternating binary masks
        for k in range(n_layers):
            if k % 2 == 0:
                m = torch.tensor([1 if i % 2 == 0 else 0 for i in range(dim)], dtype=torch.float32)
            else:
                m = torch.tensor([0 if i % 2 == 0 else 1 for i in range(dim)], dtype=torch.float32)
            masks.append(m)

        self.couplings = nn.ModuleList([AffineCoupling(dim, mask=m, hidden=hidden) for m in masks])

    def fwd(self, x):
        # x -> z
        logdet_sum = torch.zeros(x.size(0), device=x.device)
        z = x
        for c in self.couplings:
            z, logdet = c(z)
            logdet_sum = logdet_sum + logdet
        return z, logdet_sum

    def inv(self, z):
        # z -> x
        logdet_sum = torch.zeros(z.size(0), device=z.device)
        x = z
        for c in reversed(self.couplings):
            x, logdet = c.inverse(x)
            logdet_sum = logdet_sum + logdet
        return x, logdet_sum

    def log_prob(self, x):
        z, logdet = self.fwd(x)
        # standard normal base
        log_pz = -0.5 * (z ** 2 + np.log(2.0 * np.pi)).sum(dim=-1)
        return log_pz + logdet

def fit_flow_on_errors(errors_np, device, epochs=80, lr=1e-3, batch_size=256, n_layers=6, hidden=64):
    """
    Fit RealNVP on normal error vectors, return trained flow + (mean,std) used for standardizing errors.
    This preserves the fixed global torch RNG sequence used by the attack generator.
    """
    if errors_np.shape[0] == 0:
        return None, None, None

    # standardize errors for stability (fit on given normal errors)
    mu = errors_np.mean(axis=0, keepdims=True)
    sd = errors_np.std(axis=0, keepdims=True) + 1e-8
    errs_std = (errors_np - mu) / sd

    x = torch.from_numpy(errs_std.astype(np.float32))
    loader = torch.utils.data.DataLoader(
        x,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        num_workers=0
    )

    flow = RealNVP(dim=errors_np.shape[1], n_layers=n_layers, hidden=hidden).to(device)
    opt = torch.optim.Adam(flow.parameters(), lr=lr)

    flow.train()
    for ep in range(1, epochs + 1):
        total = 0.0
        n = 0
        for xb in loader:
            xb = xb.to(device)
            # maximize log_prob -> minimize negative log_prob
            nll = (-flow.log_prob(xb)).mean()
            opt.zero_grad()
            nll.backward()
            opt.step()
            total += float(nll.item()) * xb.size(0)
            n += xb.size(0)
        # optional light logging
        # if ep % 20 == 0 or ep == 1:
        #     print(f'  [flow] epoch {ep}/{epochs} nll={total/max(n,1):.4f}')
    flow.eval()
    return flow, mu, sd

def flow_scores(flow, mu, sd, errors_np, device):
    """
    Return anomaly scores = negative log-likelihood under flow (higher => more anomalous).
    """
    if errors_np.shape[0] == 0:
        return np.empty((0,), dtype=np.float64)
    errs_std = (errors_np - mu) / sd
    x = torch.from_numpy(errs_std.astype(np.float32)).to(device)
    with torch.no_grad():
        scores = (-flow.log_prob(x)).detach().cpu().numpy()
    return scores

# -----------------------
# 与原代码保持一致的结果容器初始化
# -----------------------
SUM_acc_all_ori = 0
SUM_acc_norm_ori = 0
SUM_acc_attack_ori = 0

results_sets = {
    'Type-1': {'cm': [], 'acc_all': [], 'acc_norm': [], 'acc_attack': []},
    'Type-2': {'cm': [], 'acc_all': [], 'acc_norm': [], 'acc_attack': []},
    'Type-3': {'cm': [], 'acc_all': [], 'acc_norm': [], 'acc_attack': []},
    'Type-4': {'cm': [], 'acc_all': [], 'acc_norm': [], 'acc_attack': []},
    'Type-5': {'cm': [], 'acc_all': [], 'acc_norm': [], 'acc_attack': []},
    'Type-6': {'cm': [], 'acc_all': [], 'acc_norm': [], 'acc_attack': []},
    'Type-1-ta': {'cm': [], 'acc_all': [], 'acc_norm': [], 'acc_attack': []},
    'Type-2-ta': {'cm': [], 'acc_all': [], 'acc_norm': [], 'acc_attack': []},
    'Type-3-ta': {'cm': [], 'acc_all': [], 'acc_norm': [], 'acc_attack': []},
    'Type-4-ta': {'cm': [], 'acc_all': [], 'acc_norm': [], 'acc_attack': []},
    'Type-5-ta': {'cm': [], 'acc_all': [], 'acc_norm': [], 'acc_attack': []},
    'Type-6-ta': {'cm': [], 'acc_all': [], 'acc_norm': [], 'acc_attack': []}
}

acc_all_list = []
acc_norm_list = []
acc_attack_list = []
cm_list = []

# -----------------------
# loader helper（和原代码一致）
# -----------------------
from torch.utils.data import DataLoader, TensorDataset
def make_loader(data_array, shuffle=True):
    tensor = torch.from_numpy(data_array.astype(np.float32))
    return DataLoader(
        TensorDataset(tensor, tensor),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0
    )


# =========================
# Rigorous three-way split configuration
# =========================
MODEL_TRAIN_FRACTION = 0.60
VALIDATION_FRACTION = 0.20
TEST_FRACTION = 0.20
THRESHOLD_ALPHA_CANDIDATES = sorted(set([0.01, 0.025, float(alpha), 0.075, 0.10]))
MAX_VALIDATION_FPR = 0.06
TUNING_OBJECTIVE = "maximize validation F1 subject to validation FPR <= MAX_VALIDATION_FPR"

# Save three-split results separately so old reference artifacts are not overwritten.
OUT_DIR = "saved_flow_runs_three_split"
os.makedirs(OUT_DIR, exist_ok=True)


def generate_features(charging_profiles, DT_MIN, N6_select):
    features = []
    labels = []
    selected = set(map(int, N6_select))
    for i in range(len(charging_profiles['st'])):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        d = charging_profiles['d'][i]
        st_peak = datetime(st.year, st.month, st.day, 16, 0, 0)
        et_peak = datetime(st.year, st.month, st.day, 21, 0, 0)
        mid_peak = st_peak + (et_peak - st_peak) / 2
        duration = (et - st).total_seconds() / 3600
        d_per_hour = d / duration if duration > 0 else 0
        st_offset = (st - mid_peak).total_seconds()
        et_offset = (et - mid_peak).total_seconds()
        features.append([d, d_per_hour, st_offset, et_offset, duration])
        labels.append(1 if i in selected else 0)
    return np.asarray(features, dtype=float), np.asarray(labels, dtype=int)


def deterministic_three_way_split(indices, train_frac=MODEL_TRAIN_FRACTION, val_frac=VALIDATION_FRACTION, seed=66):
    idx = np.array(sorted(map(int, indices)), dtype=int)
    rng = np.random.default_rng(int(seed))
    rng.shuffle(idx)
    n = len(idx)
    n_train = int(round(n * train_frac))
    n_val = int(round(n * val_frac))
    n_train = min(max(n_train, 0), n)
    n_val = min(max(n_val, 0), n - n_train)
    return (
        sorted(idx[:n_train].astype(int).tolist()),
        sorted(idx[n_train:n_train + n_val].astype(int).tolist()),
        sorted(idx[n_train + n_val:].astype(int).tolist()),
    )


def build_three_way_session_split(n_sessions, attack_reference_indices, seed=66):
    all_indices = list(range(int(n_sessions)))
    attack_reference = sorted(set(map(int, attack_reference_indices)))
    non_reference = [i for i in all_indices if i not in set(attack_reference)]

    ref_train, ref_validation, ref_test = deterministic_three_way_split(attack_reference, seed=seed)
    non_train, non_validation, non_test = deterministic_three_way_split(non_reference, seed=seed + 1009)

    split = {
        'train': sorted(non_train + ref_train),
        'validation': sorted(non_validation + ref_validation),
        'test': sorted(non_test + ref_test),
        'attack_reference_train': ref_train,
        'attack_reference_validation': ref_validation,
        'attack_reference_test': ref_test,
    }
    sets = {k: set(v) for k, v in split.items() if k in ['train', 'validation', 'test']}
    leakage = {
        'train_validation_overlap': len(sets['train'] & sets['validation']),
        'train_test_overlap': len(sets['train'] & sets['test']),
        'validation_test_overlap': len(sets['validation'] & sets['test']),
        'unassigned_sessions': n_sessions - len(sets['train'] | sets['validation'] | sets['test']),
    }
    if any(leakage.values()):
        raise RuntimeError(f'Three-way split leakage detected: {leakage}')
    split['leakage'] = leakage
    return split


def subset_indices(split_indices, candidate_indices):
    split_set = set(map(int, split_indices))
    return sorted(i for i in map(int, candidate_indices) if i in split_set)


def errors_from_scaled_array(model, data_scaled, device):
    if data_scaled.shape[0] == 0:
        return np.empty((0, data_scaled.shape[1]), dtype=np.float32)
    tensor = torch.from_numpy(data_scaled.astype(np.float32)).to(device)
    model.eval()
    with torch.no_grad():
        recon = model(tensor)
        return (tensor - recon).cpu().numpy()


def score_raw_arrays(model, scaler, flow, e_mu, e_sd, raw_sets, device):
    scored = {}
    for name, data in raw_sets.items():
        if data.shape[0] == 0:
            scored[name] = np.empty((0,), dtype=float)
            continue
        data_scaled = scaler.transform(data)
        err_vec = errors_from_scaled_array(model, data_scaled, device)
        scored[name] = flow_scores(flow, e_mu, e_sd, err_vec, device)
    return scored


def compute_binary_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    fpr = fp / max(fp + tn, 1)
    return {
        'cm': cm,
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'fpr': float(fpr),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }


def select_threshold_from_validation(scores_norm, validation_attack_scores):
    attack_scores = [v for v in validation_attack_scores.values() if len(v) > 0]
    attack_scores = np.concatenate(attack_scores) if attack_scores else np.empty((0,), dtype=float)
    y_true = np.concatenate([np.zeros_like(scores_norm, dtype=int), np.ones_like(attack_scores, dtype=int)])
    scores = np.concatenate([scores_norm, attack_scores])

    choices = []
    for alpha_candidate in THRESHOLD_ALPHA_CANDIDATES:
        q = 1.0 - float(alpha_candidate)
        threshold = float(np.quantile(scores_norm, q))
        y_pred = (scores > threshold).astype(int)
        metrics = compute_binary_metrics(y_true, y_pred)
        choices.append({
            'alpha': float(alpha_candidate),
            'quantile': float(q),
            'threshold': threshold,
            **metrics,
        })

    feasible = [c for c in choices if c['fpr'] <= MAX_VALIDATION_FPR]
    pool = feasible if feasible else choices
    selected = max(pool, key=lambda c: (c['f1'], c['recall'], c['precision'], -c['fpr'], c['threshold']))
    selected = dict(selected)
    selected['used_fpr_constraint'] = bool(feasible)
    selected['candidate_count'] = len(choices)
    selected['normal_validation_count'] = int(len(scores_norm))
    selected['attack_validation_count'] = int(len(attack_scores))
    selected['objective'] = TUNING_OBJECTIVE
    selected['all_candidates'] = choices
    return selected


def json_ready(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    return value


# -----------------------
# Main single-seed run with train / validation-tuning / held-out-test separation
# -----------------------
def run_seed(SEED):
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
    os.environ['NUMEXPR_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)

    os.environ['PYTHONHASHSEED'] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    device = torch.device('cpu')

    #Type-1-ta
    N1_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N1_ta_indices, ta)

    #Type-2-ta
    ta_local = datetime(2025, 9, 4, 15, 55, 00)
    N2_ta = []
    for i in range(len(d_list)):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
        et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
        ta1 = datetime.combine(st.date(), ta_local.time())
        if st <= ta1 < et:
            ta_local = ta1
            if ta_local < st_peek and et > st_peek:
                N2_ta.append(i)

    N2_ta_indices = {"N2_ta": N2_ta} # Type-2
    N2_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N2_ta_indices, ta_local)

    #Type-3-ta
    ta_local = datetime(2025, 9, 4, 15, 55, 00)
    N3_ta = []
    for i in range(len(d_list)):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
        et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
        ta1 = datetime.combine(st.date(), ta_local.time())
        if st <= ta1 < et:
            ta_local = ta1
            if ta_local < et_peek and et > et_peek:
                N3_ta.append(i)

    N3_ta_indices = {"N3_ta": N3_ta} # Type-3
    N3_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N3_ta_indices, ta_local)

    #Type-4-ta
    ta_local = datetime(2025, 9, 4, 15, 55, 00)
    N4_ta = []
    for i in range(len(d_list)):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
        et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
        ta1 = datetime.combine(st.date(), ta_local.time())
        if st <= ta1 < et:
            ta_local = ta1
            if ta_local < et_peek and et > st_peek:
                N4_ta.append(i)

    N4_ta_indices = {"N4_ta": N4_ta} # Type-4
    N4_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N4_ta_indices, ta_local)

    #Type-5-ta
    ta_local = datetime(2025, 9, 5, 15, 55, 00)
    N5_ta = []
    for i in range(len(d_list)):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
        et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
        ta1 = datetime.combine(st.date(), ta_local.time())
        if st <= ta1 < et:
            ta_local = ta1
            if ta_local < et_peek and et > st_peek:
                N5_ta.append(i)

    N5_ta_indices = {"N5_ta": N5_ta} # Type-5
    N5_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N5_ta_indices, ta_local)

    #Type-6-ta
    ta_local = datetime(2026, 9, 4, 18, 55, 00)
    N6_ta = []
    for i in range(len(d_list)):
        st = charging_profiles['st'][i]
        et = charging_profiles['et'][i]
        st_peek = datetime(st.year, st.month, st.day, 16, 0, 0)
        et_peek = datetime(st.year, st.month, st.day, 21, 0, 0)
        ta1 = datetime.combine(st.date(), ta_local.time())
        if st <= ta1 < et:
            ta_local = ta1
            if ta_local < et_peek and et > st_peek:
                N6_ta.append(i)

    N6_ta_indices = {"N6_ta": N6_ta} # Type-6
    N6_ta_profiles = launch_CMAs_ta(charging_profiles, control_params, N6_ta_indices, ta_local)

    #Type-1
    Nhd_profiles = launch_CMAs(charging_profiles, control_params, Nhd_indices)
    #Type-2
    Nhe_profiles = launch_CMAs(charging_profiles, control_params, Nhe_indices)
    #Type-3
    Nhs_profiles = launch_CMAs(charging_profiles, control_params, Nhs_indices)
    #Type-4
    N4_profiles = launch_CMAs(charging_profiles, control_params, N4_indices)
    #Type_5
    N5_profiles = launch_CMAs(charging_profiles, control_params, N5_indices)
    #Type-6
    N6_indices_select = {"N6": N6} # Type-6
    N6_profiles = launch_CMAs(charging_profiles, control_params, N6_indices_select)

    split = build_three_way_session_split(len(d_list), N6, seed=SEED)
    N_normal_train = split['train']
    N_validation = split['validation']
    N_test = split['test']
    N_attacked_train = split['attack_reference_train']
    N_attacked_validation = split['attack_reference_validation']
    N_attacked_test = split['attack_reference_test']

    charging_features, _ = generate_features(charging_profiles, DT_MIN, [])

    attack_specs = {
        'Type-1': {'profiles': Nhd_profiles, 'indices_all': list(map(int, Nhd))},
        'Type-2': {'profiles': Nhe_profiles, 'indices_all': list(map(int, Nhe))},
        'Type-3': {'profiles': Nhs_profiles, 'indices_all': list(map(int, Nhs))},
        'Type-4': {'profiles': N4_profiles, 'indices_all': list(map(int, N4))},
        'Type-5': {'profiles': N5_profiles, 'indices_all': list(map(int, N5))},
        'Type-6': {'profiles': N6_profiles, 'indices_all': list(map(int, N6))},
        'Type-1-ta': {'profiles': N1_ta_profiles, 'indices_all': list(map(int, N1_ta))},
        'Type-2-ta': {'profiles': N2_ta_profiles, 'indices_all': list(map(int, N2_ta))},
        'Type-3-ta': {'profiles': N3_ta_profiles, 'indices_all': list(map(int, N3_ta))},
        'Type-4-ta': {'profiles': N4_ta_profiles, 'indices_all': list(map(int, N4_ta))},
        'Type-5-ta': {'profiles': N5_ta_profiles, 'indices_all': list(map(int, N5_ta))},
        'Type-6-ta': {'profiles': N6_ta_profiles, 'indices_all': list(map(int, N6_ta))},
    }

    train_attack_indices = {}
    validation_attack_indices = {}
    test_attack_indices = {}
    validation_attack_sets = {}
    attack_sets = {}
    for name, spec in attack_specs.items():
        features, labels = generate_features(spec['profiles'], DT_MIN, spec['indices_all'])
        spec['features'] = features
        spec['labels'] = labels
        train_attack_indices[name] = subset_indices(N_normal_train, spec['indices_all'])
        validation_attack_indices[name] = subset_indices(N_validation, spec['indices_all'])
        test_attack_indices[name] = subset_indices(N_test, spec['indices_all'])
        validation_attack_sets[name] = features[validation_attack_indices[name], 0:5]
        attack_sets[name] = features[test_attack_indices[name], 0:5]

    print('[three-way split]', {k: len(split[k]) for k in ['train', 'validation', 'test']})
    print('[attack reference split]', {
        'train': len(N_attacked_train),
        'validation': len(N_attacked_validation),
        'test': len(N_attacked_test),
    })
    print('[leakage check]', split['leakage'])
    print('[validation attack counts]', {k: len(v) for k, v in validation_attack_indices.items()})
    print('[held-out test attack counts]', {k: len(v) for k, v in test_attack_indices.items()})

    non_attacked_train = charging_features[N_normal_train, 0:5]
    non_attacked_validation = charging_features[N_validation, 0:5]
    non_attacked_test = charging_features[N_test, 0:5]

    scaler = StandardScaler()
    non_attacked_train_scaled = scaler.fit_transform(non_attacked_train)
    non_attacked_validation_scaled = scaler.transform(non_attacked_validation)
    non_attacked_test_scaled = scaler.transform(non_attacked_test)

    train_loader = make_loader(non_attacked_train_scaled, shuffle=True)

    model = Autoencoder(input_dim=5, latent_dim=latent_dim).to(device)
    criterion = torch.nn.MSELoss(reduction='none')
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    model.train()
    for epoch in range(1, num_epochs + 1):
        total_loss = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            recon = model(x)
            loss = criterion(recon, x).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
        if epoch % 100 == 0 or epoch == 1:
            avg_loss = total_loss / len(train_loader.dataset)
            print(f'[seed {SEED}] Epoch [{epoch}/{num_epochs}] Loss: {avg_loss:.6f}')

    errors_train = errors_from_scaled_array(model, non_attacked_train_scaled, device)
    errors_validation = errors_from_scaled_array(model, non_attacked_validation_scaled, device)
    errors_test = errors_from_scaled_array(model, non_attacked_test_scaled, device)

    flow, e_mu, e_sd = fit_flow_on_errors(
        errors_np=errors_train,
        device=device,
        epochs=FLOW_EPOCHS,
        lr=FLOW_LR,
        batch_size=FLOW_BS,
        n_layers=FLOW_LAYERS,
        hidden=FLOW_HIDDEN,
    )

    if flow is None:
        return {
            'seed': int(SEED),
            'sum_acc_all': 0.0,
            'sum_acc_norm': 0.0,
            'sum_acc_attack': 0.0,
            'per_attack': {k: {'cm': None, 'acc_all': None, 'acc_norm': None, 'acc_attack': None} for k in attack_sets.keys()},
        }

    scores_validation_norm = flow_scores(flow, e_mu, e_sd, errors_validation, device)
    scores_test_norm = flow_scores(flow, e_mu, e_sd, errors_test, device)
    validation_attack_scores = score_raw_arrays(model, scaler, flow, e_mu, e_sd, validation_attack_sets, device)

    tuning_choice = select_threshold_from_validation(scores_validation_norm, validation_attack_scores)
    thresh = float(tuning_choice['threshold'])
    selected_alpha = float(tuning_choice['alpha'])
    print(
        f"[seed {SEED}] Validation-selected flow threshold={thresh:.6f} "
        f"(normal quantile={tuning_choice['quantile']:.3f}, alpha={selected_alpha:.3f}, "
        f"validation FPR={tuning_choice['fpr']:.4f}, validation F1={tuning_choice['f1']:.4f})"
    )

    y_pred_norm = (scores_test_norm > thresh).astype(int)
    y_true_norm = np.zeros_like(y_pred_norm)
    test_attack_scores = score_raw_arrays(model, scaler, flow, e_mu, e_sd, attack_sets, device)

    SUM_acc_all = 0.0
    SUM_acc_norm = 0.0
    SUM_acc_attack = 0.0
    per_attack_results = {}

    for name, scores_attack in test_attack_scores.items():
        if scores_attack.shape[0] == 0:
            per_attack_results[name] = {'cm': None, 'acc_all': None, 'acc_norm': None, 'acc_attack': None}
            continue

        y_pred_attack = (scores_attack > thresh).astype(int)
        y_true_attack = np.ones_like(y_pred_attack)
        y_true = np.concatenate([y_true_norm, y_true_attack])
        y_pred = np.concatenate([y_pred_norm, y_pred_attack])
        metrics = compute_binary_metrics(y_true, y_pred)
        acc_norm = accuracy_score(y_true_norm, y_pred_norm)
        acc_attack = accuracy_score(y_true_attack, y_pred_attack)

        SUM_acc_all += metrics['accuracy']
        SUM_acc_norm += acc_norm
        SUM_acc_attack += acc_attack

        per_attack_results[name] = {
            'cm': metrics['cm'],
            'acc_all': float(metrics['accuracy']),
            'acc_norm': float(acc_norm),
            'acc_attack': float(acc_attack),
            'precision': float(metrics['precision']),
            'recall': float(metrics['recall']),
            'f1': float(metrics['f1']),
            'fpr': float(metrics['fpr']),
        }

    all_test_attack_scores = [v for v in test_attack_scores.values() if len(v) > 0]
    all_test_attack_scores = np.concatenate(all_test_attack_scores) if all_test_attack_scores else np.empty((0,), dtype=float)
    aggregate_y_true = np.concatenate([np.zeros_like(scores_test_norm, dtype=int), np.ones_like(all_test_attack_scores, dtype=int)])
    aggregate_scores = np.concatenate([scores_test_norm, all_test_attack_scores])
    aggregate_y_pred = (aggregate_scores > thresh).astype(int)
    aggregate_test_metrics = compute_binary_metrics(aggregate_y_true, aggregate_y_pred)

    seed_dir = os.path.join(OUT_DIR, f"seed_{int(SEED)}")
    os.makedirs(seed_dir, exist_ok=True)

    detector_ckpt = {
        'seed': int(SEED),
        'split_scheme': 'three_way_train_validation_test',
        'tuning_objective': TUNING_OBJECTIVE,
        'selected_alpha': selected_alpha,
        'max_validation_fpr': float(MAX_VALIDATION_FPR),
        'ae_input_dim': 5,
        'ae_latent_dim': int(latent_dim),
        'ae_state_dict': model.state_dict(),
        'flow_dim': int(errors_train.shape[1]),
        'flow_layers': int(FLOW_LAYERS),
        'flow_hidden': int(FLOW_HIDDEN),
        'flow_state_dict': flow.state_dict(),
        'e_mu': e_mu,
        'e_sd': e_sd,
        'flow_threshold': float(thresh),
        'validation_tuning': json_ready(tuning_choice),
        'aggregate_test_metrics': json_ready(aggregate_test_metrics),
    }
    torch.save(detector_ckpt, os.path.join(seed_dir, 'detector.pt'))
    joblib.dump(scaler, os.path.join(seed_dir, 'scaler.joblib'))

    splits = {
        'model_train_indices': list(map(int, N_normal_train)),
        'validation_indices': list(map(int, N_validation)),
        'test_indices': list(map(int, N_test)),
        'attack_reference_train': list(map(int, N_attacked_train)),
        'attack_reference_validation': list(map(int, N_attacked_validation)),
        'attack_reference_test': list(map(int, N_attacked_test)),
        'leakage': split['leakage'],
        # Backward-compatible aliases for older helper scripts.
        'N_normal_train': list(map(int, N_normal_train)),
        'N_validation': list(map(int, N_validation)),
        'N_test': list(map(int, N_test)),
        'N_attacked_train': list(map(int, N_attacked_train)),
        'N_attacked_validation': list(map(int, N_attacked_validation)),
        'N_attacked_test': list(map(int, N_attacked_test)),
    }

    attacks = {}
    for name, spec in attack_specs.items():
        attacks[name] = {
            'indices_all': list(map(int, spec['indices_all'])),
            'indices_train': list(map(int, train_attack_indices[name])),
            'indices_validation': list(map(int, validation_attack_indices[name])),
            'indices_test': list(map(int, test_attack_indices[name])),
            'profiles': spec['profiles'],
        }

    meta = {
        'split_scheme': 'three_way_train_validation_test',
        'splits': splits,
        'attacks': attacks,
        'train_data': {'non_attacked_train': non_attacked_train},
        'validation_data': {
            'non_attacked_validation': non_attacked_validation,
            'validation_attack_sets': validation_attack_sets,
            'validation_attack_scores': validation_attack_scores,
        },
        'test_data': {
            'non_attacked_test': non_attacked_test,
            'attack_sets': attack_sets,
            'test_attack_scores': test_attack_scores,
        },
        'threshold_tuning': json_ready(tuning_choice),
        'aggregate_test_metrics': json_ready(aggregate_test_metrics),
        'per_attack_test_metrics': json_ready(per_attack_results),
    }
    joblib.dump(meta, os.path.join(seed_dir, 'data_and_indices.joblib'))

    summary_json = {
        'seed': int(SEED),
        'split_scheme': 'three_way_train_validation_test',
        'split_counts': {k: int(len(split[k])) for k in ['train', 'validation', 'test']},
        'attack_reference_counts': {
            'train': int(len(N_attacked_train)),
            'validation': int(len(N_attacked_validation)),
            'test': int(len(N_attacked_test)),
        },
        'validation_attack_counts': {k: int(len(v)) for k, v in validation_attack_indices.items()},
        'test_attack_counts': {k: int(len(v)) for k, v in test_attack_indices.items()},
        'threshold_tuning': json_ready(tuning_choice),
        'aggregate_test_metrics': json_ready(aggregate_test_metrics),
        'per_attack_test_metrics': json_ready(per_attack_results),
    }
    with open(os.path.join(seed_dir, 'three_split_summary.json'), 'w', encoding='utf-8') as f:
        json.dump(summary_json, f, indent=2)

    with open(os.path.join(seed_dir, 'summary.txt'), 'w', encoding='utf-8') as f:
        f.write(f"seed={int(SEED)}\n")
        f.write("split_scheme=three_way_train_validation_test\n")
        f.write(f"model_train_count={len(N_normal_train)}\n")
        f.write(f"validation_count={len(N_validation)}\n")
        f.write(f"test_count={len(N_test)}\n")
        f.write(f"selected_alpha={selected_alpha}\n")
        f.write(f"flow_threshold={float(thresh)}\n")
        f.write(f"validation_fpr={float(tuning_choice['fpr'])}\n")
        f.write(f"validation_f1={float(tuning_choice['f1'])}\n")
        f.write(f"aggregate_test_accuracy={float(aggregate_test_metrics['accuracy'])}\n")
        f.write(f"aggregate_test_precision={float(aggregate_test_metrics['precision'])}\n")
        f.write(f"aggregate_test_recall={float(aggregate_test_metrics['recall'])}\n")
        f.write(f"aggregate_test_f1={float(aggregate_test_metrics['f1'])}\n")

    return {
        'seed': int(SEED),
        'split_scheme': 'three_way_train_validation_test',
        'threshold_tuning': tuning_choice,
        'aggregate_test_metrics': aggregate_test_metrics,
        'sum_acc_all': float(SUM_acc_all),
        'sum_acc_norm': float(SUM_acc_norm),
        'sum_acc_attack': float(SUM_acc_attack),
        'per_attack': per_attack_results,
    }


print(f"Running rigorous three-way split on CPU: SEED={BEST_SEED}")
result = run_seed(BEST_SEED)

for k in results_sets.keys():
    results_sets[k]['cm'].clear()
    results_sets[k]['acc_all'].clear()
    results_sets[k]['acc_norm'].clear()
    results_sets[k]['acc_attack'].clear()

for name, stats in result['per_attack'].items():
    results_sets[name]['cm'].append(stats['cm'])
    results_sets[name]['acc_all'].append(stats['acc_all'])
    results_sets[name]['acc_norm'].append(stats['acc_norm'])
    results_sets[name]['acc_attack'].append(stats['acc_attack'])

print('[info] held-out test results_sets populated. Example lens:',
      {k: len(results_sets[k]['cm']) for k in list(results_sets.keys())[:3]})
print('[Three Split Summary]')
print('seed:', result.get('seed'))
print('selected threshold:', result['threshold_tuning']['threshold'])
print('validation FPR/F1:', result['threshold_tuning']['fpr'], result['threshold_tuning']['f1'])
print('aggregate held-out test metrics:', json_ready(result['aggregate_test_metrics']))


In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 10))
axes = axes.flatten()
best_idx = 0   # 单seed运行时，results_sets 每个列表只有一个元素

axes = axes.flatten()
for i, name in enumerate(results_sets.keys()):
    cm = results_sets[name]['cm'][best_idx]
    acc_all = results_sets[name]['acc_all'][best_idx]
    acc_norm = results_sets[name]['acc_norm'][best_idx]
    acc_attack = results_sets[name]['acc_attack'][best_idx]

    print(f"\n=== {name} ===")
    print(f"All Acc:      {acc_all*100:.2f}%")
    print(f"Norm Acc:     {acc_norm*100:.2f}%")
    print(f"Attack Acc:   {acc_attack*100:.2f}%")
    print("Confusion Matrix:")
    print(cm)

    ax = axes[i]
    ax.imshow(cm, interpolation='nearest', cmap='Blues', aspect='equal')
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Normal','Anomaly']); ax.set_yticklabels(['Normal','Anomaly'])
    for (m,n), v in np.ndenumerate(cm):
        ax.text(n, m, v, ha='center', va='center')
    ax.text(0.5, 2.3, f'N:{acc_norm*100:.1f}%', ha='center')
    ax.text(1.5, 2.3, f'A:{acc_attack*100:.1f}%', ha='center')

plt.tight_layout()
plt.show()